# Reliability Dynamics
Configuration: `stages/01_reliability_dynamics/config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
reliability_dynamics_experiment_dir = ROOT / 'experiment/research/osd/stages/01_reliability_dynamics'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

reliability_dynamics_data_paths = ExperimentPaths(ROOT, reliability_dynamics_experiment_dir / 'config/paths.json')


# BP message dynamics for OSD reliability

The mechanism cohort is deliberately event-driven: distance 5 uses only
weight-2/3 DEM-error instances and distance 9 uses only weight-4/5 instances.
The separately sampled Monte Carlo cohort retains every error weight and is
the only population used for unbiased LER estimates.


In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from itertools import product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kendalltau, pearsonr, spearmanr
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython import BPOSD
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

reliability_dynamics_config = json.loads((reliability_dynamics_experiment_dir / 'config/bp_message_dynamics_for_osd_reliability_experiments.json').read_text(encoding='utf-8'))
if reliability_dynamics_config['smoke_mode']:
    reliability_dynamics_config.update(
        levels=[2],
        event_p_values=[0.05],
        event_gap_targets_by_level={'2': 2},
        event_batch_size=40,
        event_max_shots_per_point=80,
        unbiased_p_values=[0.05],
        unbiased_shots_per_point=12,
        bp_iterations=8,
        snapshot_iterations=[1, 2, 4, 8],
        temporal_windows=[3, 5],
        instability_beta_values=[0.5],
        sign_gamma_values=[1.0],
        conflict_beta_values=[0.5],
        rank_threshold_values=[0.9],
        rank_persistence_values=[2],
        message_residual_quantiles=[0.5],
        combined_weight_values=[0.0, 1.0],
        bootstrap_repetitions=50,
        scaling_values=[1.0],
        damping_values=[0.0],
        schedule_values=['parallel'],
        iteration_values=[4, 8],
        hyperparameter_case_limit=8,
    )

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# reuse_existing_data: boolean (True or False).
# levels: non-empty.
# levels: values restricted to {2, 3}.
# primary_level: chosen from levels.
# instance_weights_by_level: {'2': [2, 3], '3': [4, 5]}.
# event_p_values, unbiased_p_values: each item is in (0, 1).
# event_batch_size: positive.
# event_max_shots_per_point: positive.
# unbiased_shots_per_point: positive.
# Expected relationship: bp_iterations >= maximum of snapshot_iterations.
# osd_orders_by_level: include a string key for every entry in levels.
# values of osd_orders_by_level: each order list is sorted, unique, and begins with [0, 1].
# temporal_windows, top_k_values: each item is positive.
# snapshot_stability_window: positive.
# damping_values: each item is in [0, 1).
# scaling_values: each item is positive.
# schedule_values: values restricted to {'parallel', 'serial'}.
# development_fraction: in (0, 1).
# bootstrap_repetitions: positive.
# message_zero_tolerance: non-negative.
experiment_name = 'reliability_dynamics_smoke' if reliability_dynamics_config['smoke_mode'] else 'reliability_dynamics'
OUT = reliability_dynamics_data_paths.run(smoke_mode=reliability_dynamics_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(reliability_dynamics_data_paths, reliability_dynamics_config, {'osd': OSD_VERSION})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(reliability_dynamics_data_paths, reliability_dynamics_config)

STATS, TRACE, CONF = (reliability_dynamics_data_paths.output('stats'), reliability_dynamics_data_paths.output('trace'), reliability_dynamics_data_paths.output('config'))

for reliability_dynamics_directory in (STATS, TRACE, CONF):
    reliability_dynamics_directory.mkdir(parents=True, exist_ok=True)

config_path = reliability_dynamics_data_paths.output('config_json')

if not config_path.is_file() or not reliability_dynamics_config['reuse_existing_data']:
    config_path.write_text(json.dumps({'config': reliability_dynamics_config}, indent=2), encoding='utf-8')

from experiment.utils import misc

def log(level, payload):
    return misc.log(level, payload, CONFIG=reliability_dynamics_config)

def stage_cached(paths):
    return misc.stage_cached(paths, CONFIG=reliability_dynamics_config)

reliability_dynamics_read_csv = misc.read_csv
reliability_dynamics_bits = misc.bits
reliability_dynamics_bit_array = misc.bit_array_variant_3

def split_for(shot_id):
    digest = sha256(f"{reliability_dynamics_config['random_seed']}:{shot_id}".encode()).digest()
    value = int.from_bytes(digest[:8], 'little') / 2 ** 64
    return 'development' if value < reliability_dynamics_config['development_fraction'] else 'test'

sns.set_theme(style='whitegrid')
log(2, {'output': str(OUT)})


## Decoder context and fixed-budget message tracing

Initialize the decoder context and fixed-budget native BP history. Keeping the full diagnostic trajectory allows snapshot quality to be studied separately from the normal early-stopping endpoint.


In [ ]:
from experiment.utils import osd_reliability

reliability_dynamics_Context = osd_reliability.Context_variant_2
reliability_dynamics_BPTrace = osd_reliability.BPTrace
reliability_dynamics_context_cache = {}
reliability_dynamics_gf2_rank = osd_reliability.gf2_rank_variant_2

def reliability_dynamics_context_for(level, p):
    return osd_reliability.context_for(
        level,
        p,
        CA_9_2_5=CA_9_2_5,
        CONTEXT_CACHE=reliability_dynamics_context_cache,
        CircuitBuilder=CircuitBuilder,
        Context=reliability_dynamics_Context,
        ErrorRate=ErrorRate,
        TriangularCode=TriangularCode,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
        gf2_rank=reliability_dynamics_gf2_rank,
    )

def reliability_dynamics_trace_min_sum(
    context,
    syndrome,
    iterations,
    scaling=1.0,
    damping=0.0,
    schedule='parallel',
    early_stop=False,
):
    return osd_reliability.trace_min_sum(context, syndrome, iterations, scaling, damping, schedule, early_stop, BPTrace=reliability_dynamics_BPTrace)

reliability_dynamics_logical_label = osd_reliability.logical_label

def reliability_dynamics_outcome(
    context,
    correction,
    syndrome,
    truth,
    error=None,
):
    return osd_reliability.outcome(context, correction, syndrome, truth, error, logical_label=reliability_dynamics_logical_label)

reliability_dynamics_ordering_proxy = osd_reliability.ordering_proxy

def reliability_dynamics_osd_from_reliability(
    context,
    syndrome,
    reliability,
    order,
):
    return osd_reliability.osd_from_reliability(context, syndrome, reliability, order, ordered_statistics_postprocess=ordered_statistics_postprocess, ordering_proxy=reliability_dynamics_ordering_proxy)

def native_bposd(context, syndrome, force_osd):
    decoder = BPOSD(
        max_iter=reliability_dynamics_config['bp_iterations'],
        ms_scaling_factor=1.0,
        osd_order=reliability_dynamics_config['osd_orders_by_level'][str(context.level)][-1],
        need_tracking=True,
    )
    decoder._init_backend(dem=context.dem)
    started = perf_counter()
    correction = np.asarray(decoder.decode_to_correction(syndrome, force_osd=force_osd), dtype=np.uint8)

    return (correction, perf_counter() - started)

def tesseract_predictions(context, syndromes):
    return osd_reliability.tesseract_predictions(context, syndromes, CONFIG=reliability_dynamics_config, Tesseract=Tesseract)


## Reliability and trajectory metrics

Define reliability ranks and temporal features before inspecting their relationship with OSD. LLR magnitude, rank stability and message convergence are separate quantities.


In [ ]:
from experiment.utils import osd_reliability

reliability_dynamics_ordinal_ranks = osd_reliability.ordinal_ranks

def percentile(values):
    return (reliability_dynamics_ordinal_ranks(values) - 1) / max(1, len(values) - 1)

def trajectory_metrics(context, trace, endpoint=None):
    endpoint = len(trace.llr) - 1 if endpoint is None else int(endpoint)
    assert 2 <= endpoint < len(trace.llr)
    eps = np.finfo(float).eps
    c2v = trace.c2v[1:endpoint + 1]
    delta = np.abs(np.diff(trace.c2v[:endpoint + 1], axis=0))
    normalized_delta = delta / (np.abs(context.prior_llr[context.edge_variables])[None, :] + eps)
    signs = np.where(np.abs(c2v) <= reliability_dynamics_config['message_zero_tolerance'], 0, np.where(c2v > 0, 1, -1))
    comparable = (signs[1:] != 0) & (signs[:-1] != 0)
    flips = ((signs[1:] != signs[:-1]) & comparable).astype(float)
    belief_delta = np.abs(np.diff(trace.llr[:endpoint + 1], axis=0))
    belief_delta /= np.abs(context.prior_llr)[None, :] + eps
    ranks = np.asarray([reliability_dynamics_ordinal_ranks(np.abs(row)) for row in trace.llr[:endpoint + 1]])
    rank_volatility = np.abs(np.diff(ranks, axis=0)).mean(axis=0)
    message_instability = np.zeros(context.H.shape[1])
    sign_flip = np.zeros_like(message_instability)
    conflict = np.zeros_like(message_instability)

    for variable, ids in enumerate(context.variable_edges):
        message_instability[variable] = normalized_delta[:, ids].mean()
        sign_flip[variable] = flips[:, ids].mean() if len(flips) else 0.0
        incoming = c2v[:, ids]
        conflict[variable] = np.mean(1 - np.abs(incoming.sum(axis=1)) / (np.abs(incoming).sum(axis=1) + eps))

    message_residual = normalized_delta.mean(axis=1)
    return {
        'message_instability': message_instability,
        'belief_instability': belief_delta.mean(axis=0),
        'sign_flip': sign_flip,
        'extrinsic_conflict': conflict,
        'rank_volatility': rank_volatility,
        'ranks': ranks,
        'message_residual': message_residual,
    }

def reliability_score(
    trace,
    metrics,
    method,
    endpoint=None,
    window=None,
    parameter=None,
    combined_weights=None,
):
    endpoint = len(trace.llr) - 1 if endpoint is None else int(endpoint)
    assert 0 <= endpoint < len(trace.llr)
    final = trace.llr[endpoint]
    window = min(endpoint + 1, reliability_dynamics_config['temporal_windows'][0] if window is None else int(window))
    recent = trace.llr[endpoint - window + 1:endpoint + 1]
    eps = np.finfo(float).eps

    if method == 'final_abs':
        return np.abs(final)
    if method == 'temporal_mean':
        return np.abs(recent.mean(axis=0))

    if method == 'temporal_median':
        return np.abs(np.median(recent, axis=0))
    if method.startswith('instability_'):
        metric = method.removeprefix('instability_')
        return np.abs(final) / (1 + float(parameter) * metrics[metric])

    if method == 'sign_consistency':
        consistency = np.abs(np.sign(recent).mean(axis=0))
        return np.abs(final) * consistency ** float(parameter)

    if method == 'snr':
        return np.abs(recent.mean(axis=0)) / (recent.std(axis=0) + eps)
    if method == 'conflict':
        return np.abs(final) / (1 + float(parameter) * metrics['extrinsic_conflict'])

    if method == 'combined':
        assert combined_weights is not None and len(combined_weights) == 4
        base = percentile(np.abs(final))
        penalties = [metrics[name] for name in ('message_instability', 'sign_flip', 'extrinsic_conflict', 'rank_volatility')]

        return base - sum((weight * percentile(value) for weight, value in zip(combined_weights, penalties)))

    raise AssertionError(method)

def top_jaccard(a, b, k):
    k = min(int(k), len(a))
    left = set(np.argsort(a, kind='stable')[:k])
    right = set(np.argsort(b, kind='stable')[:k])

    return len(left & right) / len(left | right)

reliability_dynamics_safe_corr = osd_reliability.safe_corr

def bootstrap_mean(values, seed_offset=0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(reliability_dynamics_config['random_seed'] + seed_offset)
    means = np.asarray([rng.choice(values, len(values), replace=True).mean() for _ in range(reliability_dynamics_config['bootstrap_repetitions'])])

    return (float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975)))

def aggregate_binary(frame, group_columns, value_column='logical_correct'):
    rows = []
    for key, group in frame.groupby(group_columns, dropna=False, sort=True):
        key = key if isinstance(key, tuple) else (key,)
        mean, low, high = bootstrap_mean(group[value_column].to_numpy(), len(rows))
        rows.append(dict(zip(group_columns, key)) | {
            'samples': group.shot_id.nunique(),
            'evaluations': len(group),
            value_column: mean,
            'CI95_low': low,
            'CI95_high': high,
        })

    return pd.DataFrame(rows)


## Event-driven instance cohort

Only the configured low-weight bands enter this instance-level analysis.


In [ ]:
event_paths = [reliability_dynamics_data_paths.output('event_screen_csv'), reliability_dynamics_data_paths.output('event_cohort_csv')]
if stage_cached(event_paths):
    reliability_dynamics_event_screen, event_cohort = map(reliability_dynamics_read_csv, event_paths)
    log(2, {'stage': 'event cohort', 'source': 'cache', 'cases': len(event_cohort)})
else:
    screen_rows = []
    for reliability_dynamics_level in reliability_dynamics_config['levels']:
        allowed_weights = set(reliability_dynamics_config['instance_weights_by_level'][str(reliability_dynamics_level)])
        target = reliability_dynamics_config['event_gap_targets_by_level'][str(reliability_dynamics_level)]
        group_counts = {name: 0 for name in 'ABCD'}

        for p_index, reliability_dynamics_p in enumerate(reliability_dynamics_config['event_p_values']):
            reliability_dynamics_context = reliability_dynamics_context_for(reliability_dynamics_level, reliability_dynamics_p)
            sampler = reliability_dynamics_context.dem.compile_sampler(seed=reliability_dynamics_config['random_seed'] + 10007 * reliability_dynamics_level + p_index)
            raw_offset = 0

            while raw_offset < reliability_dynamics_config['event_max_shots_per_point'] and group_counts['B'] < target:
                batch = min(reliability_dynamics_config['event_batch_size'], reliability_dynamics_config['event_max_shots_per_point'] - raw_offset)
                syndromes, truths, errors = sampler.sample(batch, return_errors=True)
                reliability_dynamics_weights = errors.sum(axis=1)
                reliability_dynamics_selected = np.flatnonzero(np.isin(reliability_dynamics_weights, list(allowed_weights)))

                if not len(reliability_dynamics_selected):
                    raw_offset += batch
                    continue
                tess_pred, tess_runtime = tesseract_predictions(reliability_dynamics_context, syndromes[reliability_dynamics_selected])

                for local_position, index in enumerate(tqdm(reliability_dynamics_selected, desc=f'screen d={reliability_dynamics_context.distance} p={reliability_dynamics_p:g}', disable=reliability_dynamics_config['print_level'] == 1)):
                    reliability_dynamics_syndrome = np.asarray(syndromes[index], dtype=np.uint8)
                    reliability_dynamics_truth = np.asarray(truths[index], dtype=np.uint8)
                    reliability_dynamics_error = np.asarray(errors[index], dtype=np.uint8)
                    assert np.array_equal(reliability_dynamics_context.H @ reliability_dynamics_error % 2, reliability_dynamics_syndrome)
                    assert np.array_equal(reliability_dynamics_logical_label(reliability_dynamics_context, reliability_dynamics_error), reliability_dynamics_truth)
                    standard, standard_runtime = native_bposd(reliability_dynamics_context, reliability_dynamics_syndrome, False)
                    forced, forced_runtime = native_bposd(reliability_dynamics_context, reliability_dynamics_syndrome, True)
                    reliability_dynamics_bp_valid, reliability_dynamics_bp_correct, bp_exact = reliability_dynamics_outcome(reliability_dynamics_context, standard if np.array_equal(reliability_dynamics_context.H @ standard % 2, reliability_dynamics_syndrome) else np.zeros_like(reliability_dynamics_error), reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                    standard_valid, reliability_dynamics_standard_correct, standard_exact = reliability_dynamics_outcome(reliability_dynamics_context, standard, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                    forced_valid, reliability_dynamics_forced_correct, forced_exact = reliability_dynamics_outcome(reliability_dynamics_context, forced, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                    tess_correct = int(np.array_equal(tess_pred[local_position], reliability_dynamics_truth))

                    if reliability_dynamics_standard_correct and reliability_dynamics_forced_correct:
                        group = 'A'
                    elif not reliability_dynamics_standard_correct and reliability_dynamics_forced_correct:
                        group = 'B'
                    elif not reliability_dynamics_standard_correct and (not reliability_dynamics_forced_correct):
                        group = 'C'
                    else:
                        group = 'other'

                    decoder = BPOSD(max_iter=reliability_dynamics_config['bp_iterations'], osd_order=0, need_tracking=True)
                    decoder._init_backend(dem=reliability_dynamics_context.dem)
                    decoder.decode_to_correction(reliability_dynamics_syndrome, force_osd=False)
                    plain = np.asarray(decoder.bp_correction, dtype=np.uint8)
                    plain_valid, plain_correct, plain_exact = reliability_dynamics_outcome(reliability_dynamics_context, plain, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                    reliability_dynamics_shot_id = f'event-l{reliability_dynamics_level}-p{p_index}-s{raw_offset + int(index)}'

                    screen_rows.append({
                        'shot_id': reliability_dynamics_shot_id,
                        'level': reliability_dynamics_level,
                        'distance': reliability_dynamics_context.distance,
                        'p': reliability_dynamics_p,
                        'split': split_for(reliability_dynamics_shot_id),
                        'error_weight': int(reliability_dynamics_error.sum()),
                        'syndrome_weight': int(reliability_dynamics_syndrome.sum()),
                        'syndrome': reliability_dynamics_bits(reliability_dynamics_syndrome),
                        'true_logical': reliability_dynamics_bits(reliability_dynamics_truth),
                        'error': reliability_dynamics_bits(reliability_dynamics_error),
                        'primary_group': group,
                        'channel_prior_llr': json.dumps(reliability_dynamics_context.prior_llr.tolist(), separators=(',', ':')),
                        'group_D': int(not plain_valid),
                        'plain_syndrome_consistent': plain_valid,
                        'plain_logical_correct': plain_correct,
                        'plain_exact_error_match': plain_exact,
                        'standard_syndrome_consistent': standard_valid,
                        'standard_bposd_correct': reliability_dynamics_standard_correct,
                        'standard_exact_error_match': standard_exact,
                        'forced_syndrome_consistent': forced_valid,
                        'forced_osd_correct': reliability_dynamics_forced_correct,
                        'forced_exact_error_match': forced_exact,
                        'tesseract_correct': tess_correct,
                        'standard_runtime_seconds': standard_runtime,
                        'forced_runtime_seconds': forced_runtime,
                        'tesseract_runtime_seconds': tess_runtime,
                    })

                    if group in group_counts:
                        group_counts[group] += 1
                    if not plain_valid:
                        group_counts['D'] += 1

                raw_offset += batch

            log(2, {
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_p,
                'screened': raw_offset,
                'eligible': len(screen_rows),
                'groups': group_counts,
            })

            if group_counts['B'] >= target:
                break

    reliability_dynamics_event_screen = pd.DataFrame(screen_rows)
    assert len(reliability_dynamics_event_screen), 'no eligible instance-level samples'
    reliability_dynamics_selected_rows = []

    for reliability_dynamics_level, level_frame in reliability_dynamics_event_screen.groupby('level', sort=True):
        target = reliability_dynamics_config['event_gap_targets_by_level'][str(int(reliability_dynamics_level))]
        gap = level_frame[level_frame.primary_group == 'B'].head(target)
        for _, anchor in gap.iterrows():
            reliability_dynamics_selected_rows.append(anchor.to_dict() | {
                'cohort_group': 'B',
                'matched_anchor': anchor.shot_id,
                'match_syndrome_weight_distance': 0,
            })

        if not len(gap):
            for group_name, reliability_dynamics_mask in {
                'A': level_frame.primary_group == 'A',
                'C': level_frame.primary_group == 'C',
                'D': level_frame.group_D == 1,
            }.items():
                for _, fallback in level_frame[reliability_dynamics_mask].head(target).iterrows():
                    reliability_dynamics_selected_rows.append(fallback.to_dict() | {
                        'cohort_group': group_name,
                        'matched_anchor': fallback.shot_id,
                        'match_syndrome_weight_distance': np.nan,
                    })

            continue

        for group_name, reliability_dynamics_mask in {
            'A': level_frame.primary_group == 'A',
            'C': level_frame.primary_group == 'C',
            'D': level_frame.group_D == 1,
        }.items():
            pool = level_frame[reliability_dynamics_mask].copy()
            used = set()
            for _, anchor in gap.iterrows():
                reliability_dynamics_candidates = pool[(pool.p == anchor.p) & (pool.error_weight == anchor.error_weight) & ~pool.shot_id.isin(used)].copy()
                if not len(reliability_dynamics_candidates):
                    continue
                reliability_dynamics_candidates['match_distance'] = np.abs(reliability_dynamics_candidates.syndrome_weight - anchor.syndrome_weight)
                chosen = reliability_dynamics_candidates.sort_values(['match_distance', 'shot_id']).iloc[0]
                used.add(chosen.shot_id)
                reliability_dynamics_selected_rows.append(chosen.drop(labels='match_distance').to_dict() | {
                    'cohort_group': group_name,
                    'matched_anchor': anchor.shot_id,
                    'match_syndrome_weight_distance': int(chosen.match_distance),
                })

    event_cohort = pd.DataFrame(reliability_dynamics_selected_rows)
    assert len(event_cohort) and set(event_cohort.error_weight.unique()) <= {2, 3, 4, 5}
    for reliability_dynamics_distance, distance_rows in event_cohort.groupby('distance'):
        shot_ids = sorted(distance_rows.shot_id.unique())
        if len(shot_ids) >= 2 and event_cohort.loc[event_cohort.distance == reliability_dynamics_distance, 'split'].nunique() < 2:
            replacement = {shot_id: 'development' if index % 2 == 0 else 'test' for index, shot_id in enumerate(shot_ids)}
            reliability_dynamics_mask = event_cohort.distance == reliability_dynamics_distance
            event_cohort.loc[reliability_dynamics_mask, 'split'] = event_cohort.loc[reliability_dynamics_mask, 'shot_id'].map(replacement)

    reliability_dynamics_event_screen.to_csv(event_paths[0], index=False)
    event_cohort.to_csv(event_paths[1], index=False)
    log(2, {
        'stage': 'event cohort',
        'source': 'experiment',
        'counts': event_cohort.groupby(['distance', 'cohort_group']).size().to_dict(),
    })

event_status = reliability_dynamics_event_screen.groupby(['distance', 'level'], as_index=False).agg(screened_eligible=('shot_id', 'size'), gap_events=('primary_group', lambda x: int((x == 'B').sum())))
event_status['target'] = event_status.level.map(lambda x: reliability_dynamics_config['event_gap_targets_by_level'][str(int(x))])
event_status['status'] = np.where(event_status.gap_events >= event_status.target, 'complete', 'insufficient_events')

event_status.to_csv(reliability_dynamics_data_paths.output('experiment_status_csv'), index=False)
event_status


## Experiments 1--4: snapshots, rank dynamics, and relevant variables

Run the tested OSD variants at multiple snapshots and inspect ranks of relevant variables. This asks whether useful soft information appears before, or survives after, a hard-decision success.


In [ ]:
def json_vector(values):
    return json.dumps(np.asarray(values).tolist(), separators=(',', ':'))

def parse_vector(value, dtype=float):
    return np.asarray(json.loads(value), dtype=dtype)

trajectory_paths = [reliability_dynamics_data_paths.output('event_case_metrics_csv'), reliability_dynamics_data_paths.output('bp_llr_trajectories_csv'), reliability_dynamics_data_paths.output('bp_rank_trajectories_csv'), reliability_dynamics_data_paths.output('bp_message_trajectories_csv'), reliability_dynamics_data_paths.output('variable_dynamics_csv')]
if stage_cached(trajectory_paths):
    event_cases, llr_trajectories, rank_trajectories, message_trajectories, variable_dynamics = (reliability_dynamics_read_csv(path) for path in trajectory_paths)
    log(2, {'stage': 'event trajectories', 'source': 'cache', 'cases': len(event_cases)})
else:
    case_rows, llr_rows, reliability_dynamics_rank_rows, message_rows, reliability_dynamics_variable_rows = ([], [], [], [], [])
    unique_cases = event_cohort.drop_duplicates('shot_id')
    for reliability_dynamics_row in tqdm(unique_cases.itertuples(index=False), total=len(unique_cases), desc='event trajectories', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context = reliability_dynamics_context_for(int(reliability_dynamics_row.level), float(reliability_dynamics_row.p))
        reliability_dynamics_syndrome = reliability_dynamics_bit_array(reliability_dynamics_row.syndrome, reliability_dynamics_context.H.shape[0])
        reliability_dynamics_truth = reliability_dynamics_bit_array(reliability_dynamics_row.true_logical, reliability_dynamics_context.L.shape[0])
        reliability_dynamics_error = reliability_dynamics_bit_array(reliability_dynamics_row.error, reliability_dynamics_context.H.shape[1])
        expected_weights = set(reliability_dynamics_config['instance_weights_by_level'][str(reliability_dynamics_context.level)])
        assert int(reliability_dynamics_error.sum()) in expected_weights
        assert np.array_equal(reliability_dynamics_context.H @ reliability_dynamics_error % 2, reliability_dynamics_syndrome)
        assert np.array_equal(reliability_dynamics_logical_label(reliability_dynamics_context, reliability_dynamics_error), reliability_dynamics_truth)
        reliability_dynamics_trace = reliability_dynamics_trace_min_sum(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_config['bp_iterations'])
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        reliability_dynamics_endpoint = reliability_dynamics_trace.first_valid if reliability_dynamics_trace.first_valid is not None else reliability_dynamics_config['bp_iterations']
        forced, _ = native_bposd(reliability_dynamics_context, reliability_dynamics_syndrome, True)
        reliability_dynamics_repair = reliability_dynamics_trace.hard[reliability_dynamics_endpoint] ^ forced if int(reliability_dynamics_row.forced_osd_correct) else np.zeros_like(forced)
        case_rows.append({
            'shot_id': reliability_dynamics_row.shot_id,
            'level': reliability_dynamics_context.level,
            'distance': reliability_dynamics_context.distance,
            'p': reliability_dynamics_context.p,
            'split': reliability_dynamics_row.split,
            'syndrome': reliability_dynamics_row.syndrome,
            'true_logical': reliability_dynamics_row.true_logical,
            'error': reliability_dynamics_row.error,
            'error_weight': int(reliability_dynamics_error.sum()),
            'syndrome_weight': int(reliability_dynamics_syndrome.sum()),
            'channel_prior_llr': json_vector(reliability_dynamics_context.prior_llr),
            'first_valid_iteration': reliability_dynamics_trace.first_valid,
            'benchmark_iteration': reliability_dynamics_endpoint,
            'benchmark_syndrome_consistent': int(reliability_dynamics_trace.valid[reliability_dynamics_endpoint]),
            'benchmark_logical_correct': reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_trace.hard[reliability_dynamics_endpoint], reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[1],
            'benchmark_exact_error_match': reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_trace.hard[reliability_dynamics_endpoint], reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[2],
            'forced_osd_correct': int(reliability_dynamics_row.forced_osd_correct),
            'repair_support_size': int(reliability_dynamics_repair.sum()),
            'trajectory_runtime_seconds': reliability_dynamics_trace.runtime_seconds[-1],
        })

        for reliability_dynamics_iteration in range(len(reliability_dynamics_trace.llr)):
            llr_rows.append({
                'shot_id': reliability_dynamics_row.shot_id,
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_context.p,
                'iteration': reliability_dynamics_iteration,
                'relative_iteration': np.nan if reliability_dynamics_trace.first_valid is None else reliability_dynamics_iteration - reliability_dynamics_trace.first_valid,
                'llr': json_vector(reliability_dynamics_trace.llr[reliability_dynamics_iteration]),
                'hard_decision': reliability_dynamics_bits(reliability_dynamics_trace.hard[reliability_dynamics_iteration]),
                'syndrome_consistent': int(reliability_dynamics_trace.valid[reliability_dynamics_iteration]),
                'logical_correct': reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_trace.hard[reliability_dynamics_iteration], reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[1],
                'exact_error_match': reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_trace.hard[reliability_dynamics_iteration], reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[2],
                'runtime_seconds': reliability_dynamics_trace.runtime_seconds[reliability_dynamics_iteration],
            })
            reliability_dynamics_ranks = metrics['ranks'][reliability_dynamics_iteration]
            reliability_dynamics_rank_rows.append({
                'shot_id': reliability_dynamics_row.shot_id,
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_context.p,
                'iteration': reliability_dynamics_iteration,
                'relative_iteration': np.nan if reliability_dynamics_trace.first_valid is None else reliability_dynamics_iteration - reliability_dynamics_trace.first_valid,
                'ranks': json_vector(reliability_dynamics_ranks),
                'kendall_previous': np.nan if reliability_dynamics_iteration == 0 else reliability_dynamics_safe_corr(metrics['ranks'][reliability_dynamics_iteration - 1], reliability_dynamics_ranks, 'spearman') if len(reliability_dynamics_ranks) < 2 else float(kendalltau(metrics['ranks'][reliability_dynamics_iteration - 1], reliability_dynamics_ranks).statistic),
                **{f'top_{min(k, len(reliability_dynamics_ranks))}_jaccard_previous': np.nan if reliability_dynamics_iteration == 0 else top_jaccard(np.abs(reliability_dynamics_trace.llr[reliability_dynamics_iteration - 1]), np.abs(reliability_dynamics_trace.llr[reliability_dynamics_iteration]), k) for k in reliability_dynamics_config['top_k_values']},
            })
            message_rows.append({
                'shot_id': reliability_dynamics_row.shot_id,
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_context.p,
                'iteration': reliability_dynamics_iteration,
                'c2v': json_vector(reliability_dynamics_trace.c2v[reliability_dynamics_iteration]),
                'v2c': json_vector(reliability_dynamics_trace.v2c[reliability_dynamics_iteration]),
            })

        for reliability_dynamics_variable in range(reliability_dynamics_context.H.shape[1]):
            reliability_dynamics_variable_rows.append({
                'shot_id': reliability_dynamics_row.shot_id,
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_context.p,
                'variable_id': reliability_dynamics_variable,
                'true_error': int(reliability_dynamics_error[reliability_dynamics_variable]),
                'repair_relevant': int(reliability_dynamics_repair[reliability_dynamics_variable]),
                'final_rank': int(metrics['ranks'][-1, reliability_dynamics_variable]),
                'rank_volatility': metrics['rank_volatility'][reliability_dynamics_variable],
                'message_instability': metrics['message_instability'][reliability_dynamics_variable],
                'belief_instability': metrics['belief_instability'][reliability_dynamics_variable],
                'sign_flip': metrics['sign_flip'][reliability_dynamics_variable],
                'extrinsic_conflict': metrics['extrinsic_conflict'][reliability_dynamics_variable],
            })

    event_cases = pd.DataFrame(case_rows)
    llr_trajectories = pd.DataFrame(llr_rows)
    rank_trajectories = pd.DataFrame(reliability_dynamics_rank_rows)
    message_trajectories = pd.DataFrame(message_rows)
    variable_dynamics = pd.DataFrame(reliability_dynamics_variable_rows)

    for reliability_dynamics_frame, reliability_dynamics_path in zip((event_cases, llr_trajectories, rank_trajectories, message_trajectories, variable_dynamics), trajectory_paths):
        reliability_dynamics_frame.to_csv(reliability_dynamics_path, index=False)
    log(2, {'stage': 'event trajectories', 'source': 'experiment', 'cases': len(event_cases)})

MEMBERSHIP = event_cohort[['shot_id', 'cohort_group', 'matched_anchor']].copy()
case_lookup = event_cases.set_index('shot_id')
llr_lookup = {(str(row.shot_id), int(row.iteration)): parse_vector(row.llr) for row in llr_trajectories.itertuples(index=False)}
rank_lookup = {(str(row.shot_id), int(row.iteration)): parse_vector(row.ranks, int) for row in rank_trajectories.itertuples(index=False)}
snapshot_path = reliability_dynamics_data_paths.output('osd_snapshot_trials_csv')

if stage_cached([snapshot_path]):
    osd_snapshot_trials = reliability_dynamics_read_csv(snapshot_path)
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_case in tqdm(event_cases.itertuples(index=False), total=len(event_cases), desc='OSD snapshots', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context = reliability_dynamics_context_for(int(reliability_dynamics_case.level), float(reliability_dynamics_case.p))
        reliability_dynamics_syndrome = reliability_dynamics_bit_array(reliability_dynamics_case.syndrome, reliability_dynamics_context.H.shape[0])
        reliability_dynamics_truth = reliability_dynamics_bit_array(reliability_dynamics_case.true_logical, reliability_dynamics_context.L.shape[0])
        reliability_dynamics_error = reliability_dynamics_bit_array(reliability_dynamics_case.error, reliability_dynamics_context.H.shape[1])

        for reliability_dynamics_iteration in reliability_dynamics_config['snapshot_iterations']:
            llr = llr_lookup[reliability_dynamics_case.shot_id, reliability_dynamics_iteration]
            for reliability_dynamics_order in reliability_dynamics_config['osd_orders_by_level'][str(reliability_dynamics_context.level)]:
                started = perf_counter()
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, np.abs(llr), reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_case.shot_id,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_context.p,
                    'iteration': reliability_dynamics_iteration,
                    'relative_iteration': np.nan if pd.isna(reliability_dynamics_case.first_valid_iteration) else reliability_dynamics_iteration - int(reliability_dynamics_case.first_valid_iteration),
                    'osd_order': reliability_dynamics_order,
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'exact_error_match': exact,
                    'runtime_seconds': perf_counter() - started,
                })

    osd_snapshot_trials = pd.DataFrame(reliability_dynamics_rows)
    osd_snapshot_trials.to_csv(snapshot_path, index=False)

snapshot_grouped = osd_snapshot_trials.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')
absolute_success = aggregate_binary(snapshot_grouped, ['distance', 'cohort_group', 'osd_order', 'iteration'])
absolute_success['alignment'] = 'absolute'
absolute_success['relative_iteration'] = np.nan
relative_input = snapshot_grouped[snapshot_grouped.relative_iteration.notna()].copy()
relative_success = aggregate_binary(relative_input, ['distance', 'cohort_group', 'osd_order', 'relative_iteration'])
relative_success['alignment'] = 'first_valid'
relative_success['iteration'] = np.nan
osd_success_vs_iteration = pd.concat([absolute_success, relative_success], ignore_index=True, sort=False)

osd_success_vs_iteration.to_csv(reliability_dynamics_data_paths.output('osd_success_vs_iteration_csv'), index=False)
rank_grouped = rank_trajectories.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')
reliability_dynamics_rank_rows = []

for (reliability_dynamics_distance, group, reliability_dynamics_iteration), reliability_dynamics_frame in rank_grouped.groupby(['distance', 'cohort_group', 'iteration'], sort=True):
    if reliability_dynamics_iteration == 0:
        continue
    for metric in ['kendall_previous'] + [column for column in reliability_dynamics_frame if column.endswith('jaccard_previous')]:
        reliability_dynamics_mean, reliability_dynamics_low, reliability_dynamics_high = bootstrap_mean(reliability_dynamics_frame[metric])
        reliability_dynamics_rank_rows.append({
            'distance': reliability_dynamics_distance,
            'cohort_group': group,
            'iteration': reliability_dynamics_iteration,
            'metric': metric,
            'mean': reliability_dynamics_mean,
            'CI95_low': reliability_dynamics_low,
            'CI95_high': reliability_dynamics_high,
            'samples': reliability_dynamics_frame.shot_id.nunique(),
        })

for (reliability_dynamics_distance, group), reliability_dynamics_frame in variable_dynamics.merge(MEMBERSHIP, on='shot_id').groupby(['distance', 'cohort_group'], sort=True):
    sample_values = reliability_dynamics_frame.groupby('shot_id').rank_volatility.mean()
    reliability_dynamics_mean, reliability_dynamics_low, reliability_dynamics_high = bootstrap_mean(sample_values)
    reliability_dynamics_rank_rows.append({
        'distance': reliability_dynamics_distance,
        'cohort_group': group,
        'iteration': -1,
        'metric': 'mean_rank_volatility',
        'mean': reliability_dynamics_mean,
        'CI95_low': reliability_dynamics_low,
        'CI95_high': reliability_dynamics_high,
        'samples': len(sample_values),
    })

reliability_rank_dynamics = pd.DataFrame(reliability_dynamics_rank_rows)
reliability_rank_dynamics.to_csv(reliability_dynamics_data_paths.output('reliability_rank_dynamics_csv'), index=False)
correlation_rows = []
variable_grouped = variable_dynamics.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')

for (reliability_dynamics_distance, group, reliability_dynamics_shot_id), reliability_dynamics_frame in variable_grouped.groupby(['distance', 'cohort_group', 'shot_id'], sort=True):
    reliability_dynamics_k = min(reliability_dynamics_config['top_k_values'][1], len(reliability_dynamics_frame))
    volatility = reliability_dynamics_frame.rank_volatility.to_numpy()
    for metric in ['message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict']:
        reliability_dynamics_values = reliability_dynamics_frame[metric].to_numpy()
        high_a = set(np.argsort(reliability_dynamics_values)[-reliability_dynamics_k:])
        high_b = set(np.argsort(volatility)[-reliability_dynamics_k:])
        correlation_rows.append({
            'shot_id': reliability_dynamics_shot_id,
            'distance': reliability_dynamics_distance,
            'cohort_group': group,
            'metric': metric,
            'spearman': reliability_dynamics_safe_corr(reliability_dynamics_values, volatility, 'spearman'),
            'pearson': reliability_dynamics_safe_corr(reliability_dynamics_values, volatility, 'pearson'),
            'top_k': reliability_dynamics_k,
            'top_k_jaccard': len(high_a & high_b) / len(high_a | high_b),
        })

correlations = pd.DataFrame(correlation_rows)
instability_rank_correlation = correlations.groupby(['distance', 'cohort_group', 'metric', 'top_k'], as_index=False).agg(
    samples=('shot_id', 'nunique'),
    spearman=('spearman', 'mean'),
    pearson=('pearson', 'mean'),
    top_k_jaccard=('top_k_jaccard', 'mean'),
)
instability_rank_correlation.to_csv(reliability_dynamics_data_paths.output('instability_rank_correlation_csv'), index=False)
relevant_rows = []
outcome_rows = []

for reliability_dynamics_case in event_cases.itertuples(index=False):
    comparison_group = 'BP_syndrome_invalid' if not int(reliability_dynamics_case.benchmark_syndrome_consistent) else 'BP_correct' if int(reliability_dynamics_case.benchmark_logical_correct) else 'BP_wrong_ForcedOSD_correct' if int(reliability_dynamics_case.forced_osd_correct) else 'BP_wrong_ForcedOSD_wrong'
    outcome_rows.append({'shot_id': reliability_dynamics_case.shot_id, 'comparison_group': comparison_group})

relevant_input = variable_dynamics.merge(pd.DataFrame(outcome_rows), on='shot_id', validate='many_to_one')
for (reliability_dynamics_distance, comparison_group, reliability_dynamics_shot_id), reliability_dynamics_frame in relevant_input.groupby(['distance', 'comparison_group', 'shot_id'], sort=True):
    assert len(reliability_dynamics_frame) == reliability_dynamics_frame.variable_id.nunique()
    for reliability_dynamics_iteration in reliability_dynamics_config['snapshot_iterations']:
        reliability_dynamics_ranks = rank_lookup[str(reliability_dynamics_shot_id), reliability_dynamics_iteration]
        for relevant_name, column in [('true_error', 'true_error'), ('repair', 'repair_relevant')]:
            relevant = reliability_dynamics_frame[column].to_numpy(dtype=bool)
            if not relevant.any():
                continue
            record = {
                'shot_id': reliability_dynamics_shot_id,
                'distance': reliability_dynamics_distance,
                'comparison_group': comparison_group,
                'iteration': reliability_dynamics_iteration,
                'relevant_set': relevant_name,
                'relevant_count': int(relevant.sum()),
                'mean_rank': float(reliability_dynamics_ranks[relevant].mean()),
            }

            for reliability_dynamics_k in reliability_dynamics_config['top_k_values']:
                k_eff = min(reliability_dynamics_k, len(reliability_dynamics_ranks))
                record[f'top_{k_eff}_coverage'] = float(np.mean(reliability_dynamics_ranks[relevant] <= k_eff))

            relevant_rows.append(record)

relevant_trials = pd.DataFrame(relevant_rows)
aggregate_map = {'samples': ('shot_id', 'nunique'), 'mean_rank': ('mean_rank', 'mean')}
for column in relevant_trials.columns:
    if column.startswith('top_'):
        aggregate_map[column] = (column, 'mean')

relevant_variable_ranks = relevant_trials.groupby(['distance', 'comparison_group', 'iteration', 'relevant_set'], as_index=False).agg(**aggregate_map)
relevant_variable_ranks.to_csv(reliability_dynamics_data_paths.output('relevant_variable_ranks_csv'), index=False)
osd_success_vs_iteration.head()


## Experiments 5--11 and 14: dynamics-aware reliability

Compare the dynamics-aware reliability constructions with the baseline ordering under the configured OSD budget. A feature association matters only if the resulting ordering improves decoding on the evaluated inputs.


In [ ]:
reliability_dynamics_trace_cache = {}

def trace_from_csv(case):
    shot_id = str(case.shot_id)
    if shot_id in reliability_dynamics_trace_cache:
        return reliability_dynamics_trace_cache[shot_id]
    llr_frame = llr_trajectories[llr_trajectories.shot_id == shot_id].sort_values('iteration')
    message_frame = message_trajectories[message_trajectories.shot_id == shot_id].sort_values('iteration')
    assert len(llr_frame) == len(message_frame) == reliability_dynamics_config['bp_iterations'] + 1
    llr = np.stack([parse_vector(value) for value in llr_frame.llr])
    hard = np.stack([reliability_dynamics_bit_array(value, llr.shape[1]) for value in llr_frame.hard_decision])
    valid = llr_frame.syndrome_consistent.to_numpy(dtype=bool)
    c2v = np.stack([parse_vector(value) for value in message_frame.c2v])
    v2c = np.stack([parse_vector(value) for value in message_frame.v2c])
    first = None if pd.isna(case.first_valid_iteration) else int(case.first_valid_iteration)
    trace = reliability_dynamics_BPTrace(c2v, v2c, llr, hard, valid, llr_frame.runtime_seconds.to_numpy(dtype=float), first)
    reliability_dynamics_trace_cache[shot_id] = trace

    return trace

def case_material(case):
    context = reliability_dynamics_context_for(int(case.level), float(case.p))
    return (context, reliability_dynamics_bit_array(case.syndrome, context.H.shape[0]), reliability_dynamics_bit_array(case.true_logical, context.L.shape[0]), reliability_dynamics_bit_array(case.error, context.H.shape[1]), trace_from_csv(case))

def reliability_variants(trace, metrics):
    variants = [('final_abs', 'final', None, None, None)]
    first_endpoint = trace.first_valid if trace.first_valid is not None else len(trace.llr) - 1
    for alignment, endpoint in [('final', len(trace.llr) - 1), ('first_valid', first_endpoint)]:
        for window in reliability_dynamics_config['temporal_windows']:
            variants.extend([('temporal_mean', alignment, window, None, None), ('temporal_median', alignment, window, None, None), ('snr', alignment, window, None, None)])

    for metric in ['message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict']:
        for beta in reliability_dynamics_config['instability_beta_values']:
            variants.append((f'instability_{metric}', 'final', None, beta, None))

    for gamma in reliability_dynamics_config['sign_gamma_values']:
        variants.append(('sign_consistency', 'final', reliability_dynamics_config['temporal_windows'][-1], gamma, None))
    for beta in reliability_dynamics_config['conflict_beta_values']:
        variants.append(('conflict', 'final', None, beta, None))

    return variants

ablation_path = reliability_dynamics_data_paths.output('reliability_ablation_trials_csv')
if stage_cached([ablation_path]):
    ablation_trials = reliability_dynamics_read_csv(ablation_path)
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_case in tqdm(event_cases.itertuples(index=False), total=len(event_cases), desc='reliability ablations', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, reliability_dynamics_trace = case_material(reliability_dynamics_case)
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        for reliability_dynamics_method, reliability_dynamics_alignment, window, parameter, unused in reliability_variants(reliability_dynamics_trace, metrics):
            reliability_dynamics_endpoint = reliability_dynamics_trace.first_valid if reliability_dynamics_alignment == 'first_valid' and reliability_dynamics_trace.first_valid is not None else len(reliability_dynamics_trace.llr) - 1
            reliability_dynamics_score = reliability_score(reliability_dynamics_trace, metrics, reliability_dynamics_method, endpoint=reliability_dynamics_endpoint, window=window, parameter=parameter)
            for reliability_dynamics_order in [0, 1]:
                started = perf_counter()
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_score, reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_case.shot_id,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_context.p,
                    'split': reliability_dynamics_case.split,
                    'method': reliability_dynamics_method,
                    'alignment': reliability_dynamics_alignment,
                    'window': window,
                    'parameter': parameter,
                    'osd_order': reliability_dynamics_order,
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'exact_error_match': exact,
                    'runtime_seconds': perf_counter() - started,
                })

    ablation_trials = pd.DataFrame(reliability_dynamics_rows)
    ablation_trials.to_csv(ablation_path, index=False)

ablation_grouped = ablation_trials.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')

def write_ablation(method_selector, output_name, group_columns):
    selected = ablation_grouped[method_selector(ablation_grouped.method)].copy()
    result = aggregate_binary(selected, group_columns)
    result.to_csv(reliability_dynamics_data_paths.output('output_name', output_name=output_name), index=False)

    return result

temporal_mean_reliability = write_ablation(lambda x: x == 'temporal_mean', 'temporal_mean_reliability.csv', ['distance', 'cohort_group', 'split', 'alignment', 'window', 'osd_order'])
temporal_median_reliability = write_ablation(lambda x: x == 'temporal_median', 'temporal_median_reliability.csv', ['distance', 'cohort_group', 'split', 'alignment', 'window', 'osd_order'])
instability_penalty_ablation = write_ablation(lambda x: x.str.startswith('instability_'), 'instability_penalty_ablation.csv', ['distance', 'cohort_group', 'split', 'method', 'parameter', 'osd_order'])
sign_consistency_ablation = write_ablation(lambda x: x == 'sign_consistency', 'sign_consistency_ablation.csv', ['distance', 'cohort_group', 'split', 'window', 'parameter', 'osd_order'])
snr_reliability_ablation = write_ablation(lambda x: x == 'snr', 'snr_reliability_ablation.csv', ['distance', 'cohort_group', 'split', 'alignment', 'window', 'osd_order'])
conflict_reliability_ablation = write_ablation(lambda x: x == 'conflict', 'conflict_reliability_ablation.csv', ['distance', 'cohort_group', 'split', 'parameter', 'osd_order'])

def choose_ablation_parameters(distance, family):
    base = ablation_grouped[(ablation_grouped.distance == distance) & (ablation_grouped.osd_order == 1)].copy()
    development = base[base.split == 'development'].copy()
    if not len(development):
        development = base
    candidates = development[development.cohort_group == 'B'].copy()

    if not len(candidates):
        candidates = development
    if family == 'instability':
        candidates = candidates[candidates.method.str.startswith('instability_')]
    else:
        candidates = candidates[candidates.method == family]
    assert len(candidates), (distance, family)
    keys = ['method', 'alignment', 'window', 'parameter']
    summary = candidates.groupby(keys, dropna=False, as_index=False).logical_correct.mean()
    summary = summary.sort_values(['logical_correct', 'method', 'alignment', 'window', 'parameter'], ascending=[False, True, True, True, True], na_position='first')

    return summary.iloc[0].to_dict()

selection_rows = []
for reliability_dynamics_distance in sorted(event_cases.distance.unique()):
    for family in ['temporal_mean', 'temporal_median', 'instability', 'sign_consistency', 'snr', 'conflict']:
        reliability_dynamics_selected = choose_ablation_parameters(reliability_dynamics_distance, family)
        selection_rows.append({
            'distance': reliability_dynamics_distance,
            'family': family,
            **reliability_dynamics_selected,
        })

selected_parameters = pd.DataFrame(selection_rows)

def selected_for(distance, family):
    rows = selected_parameters[(selected_parameters.distance == distance) & (selected_parameters.family == family)]
    assert len(rows) == 1
    return rows.iloc[0]

combined_path = reliability_dynamics_data_paths.output('combined_reliability_development_trials_csv')
if stage_cached([combined_path]):
    combined_trials = reliability_dynamics_read_csv(combined_path)
else:
    reliability_dynamics_rows = []
    reliability_dynamics_development = event_cases[event_cases.split == 'development']
    for reliability_dynamics_case in tqdm(reliability_dynamics_development.itertuples(index=False), total=len(reliability_dynamics_development), desc='combined reliability', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, reliability_dynamics_trace = case_material(reliability_dynamics_case)
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        for reliability_dynamics_weights in product(reliability_dynamics_config['combined_weight_values'], repeat=4):
            reliability_dynamics_score = reliability_score(reliability_dynamics_trace, metrics, 'combined', combined_weights=reliability_dynamics_weights)
            reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_score, 1)
            reliability_dynamics_rows.append({
                'shot_id': reliability_dynamics_case.shot_id,
                'distance': reliability_dynamics_context.distance,
                'weights': json_vector(reliability_dynamics_weights),
                'logical_correct': reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[1],
            })

    combined_trials = pd.DataFrame(reliability_dynamics_rows)
    combined_trials.to_csv(combined_path, index=False)

for reliability_dynamics_distance, reliability_dynamics_frame in combined_trials.merge(MEMBERSHIP, on='shot_id').groupby('distance', sort=True):
    gap = reliability_dynamics_frame[reliability_dynamics_frame.cohort_group == 'B']
    tune = gap if len(gap) else reliability_dynamics_frame
    summary = tune.groupby('weights', as_index=False).logical_correct.mean().sort_values(['logical_correct', 'weights'], ascending=[False, True])
    selected_parameters.loc[len(selected_parameters)] = {
        'distance': reliability_dynamics_distance,
        'family': 'combined',
        'method': 'combined',
        'alignment': 'final',
        'window': np.nan,
        'parameter': summary.iloc[0].weights,
        'logical_correct': summary.iloc[0].logical_correct,
    }

selected_parameters.to_csv(reliability_dynamics_data_paths.output('selected_reliability_parameters_csv'), index=False)

def selected_score(
    distance,
    family,
    trace,
    metrics,
):
    selected = selected_for(distance, family)
    if family == 'combined':
        return reliability_score(trace, metrics, 'combined', combined_weights=parse_vector(selected.parameter))
    endpoint = trace.first_valid if selected.alignment == 'first_valid' and trace.first_valid is not None else len(trace.llr) - 1
    window = None if pd.isna(selected.window) else int(selected.window)
    parameter = None if pd.isna(selected.parameter) else float(selected.parameter)

    return reliability_score(trace, metrics, selected.method, endpoint=endpoint, window=window, parameter=parameter)

benchmark_path = reliability_dynamics_data_paths.output('event_reliability_method_trials_csv')
if stage_cached([benchmark_path]):
    event_method_trials = reliability_dynamics_read_csv(benchmark_path)
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_case in tqdm(event_cases[event_cases.split == 'test'].itertuples(index=False), total=(event_cases.split == 'test').sum(), desc='held-out reliability methods', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, reliability_dynamics_trace = case_material(reliability_dynamics_case)
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        reliability_dynamics_scores = {'final_abs': np.abs(reliability_dynamics_trace.llr[-1])}

        for family in ['temporal_mean', 'temporal_median', 'instability', 'sign_consistency', 'snr', 'conflict', 'combined']:
            reliability_dynamics_scores[family] = selected_score(reliability_dynamics_context.distance, family, reliability_dynamics_trace, metrics)
        for reliability_dynamics_method, reliability_dynamics_score in reliability_dynamics_scores.items():
            for reliability_dynamics_order in [0, 1]:
                started = perf_counter()
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_score, reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_case.shot_id,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_context.p,
                    'population': 'event_test',
                    'method': reliability_dynamics_method,
                    'osd_order': reliability_dynamics_order,
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'logical_error': 1 - reliability_dynamics_correct,
                    'exact_error_match': exact,
                    'runtime_seconds': perf_counter() - started,
                })

    event_method_trials = pd.DataFrame(reliability_dynamics_rows)
    event_method_trials.to_csv(benchmark_path, index=False)

event_method_grouped = event_method_trials.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')


## Experiments 12--13: handoff rules and snapshot selection

Test handoff and snapshot-selection rules against their diagnostic oracle opportunity. Runtime choices must use available trajectory information rather than the true outcome.


In [ ]:
def first_persistent(
    values,
    predicate,
    persistence,
    default,
):
    run = 0
    for index, value in enumerate(values, start=1):
        run = run + 1 if predicate(value) else 0
        if run >= persistence:
            return index

    return default

def handoff_endpoints(trace, metrics, message_thresholds):
    endpoint_default = len(trace.llr) - 1
    ranks = metrics['ranks']
    kendall = np.asarray([float(kendalltau(ranks[t - 1], ranks[t]).statistic) for t in range(1, len(ranks))])
    endpoints = {
        'first_syndrome_consistent': trace.first_valid if trace.first_valid is not None else endpoint_default,
    }

    for iteration in reliability_dynamics_config['iteration_values']:
        endpoints[f'fixed_{iteration}'] = min(iteration, endpoint_default)
    for quantile_name, threshold in message_thresholds.items():
        endpoints[f'message_residual_q{quantile_name}'] = first_persistent(
            metrics['message_residual'],
            lambda value,
            bound=threshold: value <= bound,
            1,
            endpoint_default,
        )

    for threshold in reliability_dynamics_config['rank_threshold_values']:
        for persistence in reliability_dynamics_config['rank_persistence_values']:
            rank_endpoint = first_persistent(
                kendall,
                lambda value,
                bound=threshold: value >= bound,
                persistence,
                endpoint_default,
            )
            endpoints[f'rank_{threshold:g}_M{persistence}'] = rank_endpoint
            valid_endpoint = endpoint_default
            run = 0

            for t in range(1, len(trace.llr)):
                run = run + 1 if trace.valid[t] and kendall[t - 1] >= threshold else 0
                if run >= persistence:
                    valid_endpoint = t
                    break

            endpoints[f'syndrome_rank_{threshold:g}_M{persistence}'] = valid_endpoint

    return endpoints

development_residuals = {}
for reliability_dynamics_distance, reliability_dynamics_frame in event_cases[event_cases.split == 'development'].groupby('distance'):
    reliability_dynamics_values = []
    for reliability_dynamics_case in reliability_dynamics_frame.itertuples(index=False):
        reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, reliability_dynamics_trace = case_material(reliability_dynamics_case)
        reliability_dynamics_values.extend(trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)['message_residual'].tolist())

    development_residuals[int(reliability_dynamics_distance)] = {str(q): float(np.quantile(reliability_dynamics_values, q)) for q in reliability_dynamics_config['message_residual_quantiles']}

for reliability_dynamics_distance in event_cases.distance.unique():
    if int(reliability_dynamics_distance) not in development_residuals:
        development_residuals[int(reliability_dynamics_distance)] = {str(q): np.inf for q in reliability_dynamics_config['message_residual_quantiles']}

handoff_path = reliability_dynamics_data_paths.output('handoff_rule_trials_csv')
if stage_cached([handoff_path]):
    handoff_trials = reliability_dynamics_read_csv(handoff_path)
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_case in tqdm(event_cases.itertuples(index=False), total=len(event_cases), desc='handoff rules', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, reliability_dynamics_trace = case_material(reliability_dynamics_case)
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        endpoints = handoff_endpoints(reliability_dynamics_trace, metrics, development_residuals[reliability_dynamics_context.distance])

        for reliability_dynamics_rule, reliability_dynamics_endpoint in endpoints.items():
            reliability_dynamics_score = np.abs(reliability_dynamics_trace.llr[reliability_dynamics_endpoint])
            for reliability_dynamics_order in [0, 1]:
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_score, reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_case.shot_id,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_context.p,
                    'split': reliability_dynamics_case.split,
                    'rule': reliability_dynamics_rule,
                    'endpoint': reliability_dynamics_endpoint,
                    'osd_order': reliability_dynamics_order,
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'logical_error': 1 - reliability_dynamics_correct,
                    'exact_error_match': exact,
                    'bp_runtime_seconds': reliability_dynamics_trace.runtime_seconds[reliability_dynamics_endpoint],
                })

    handoff_trials = pd.DataFrame(reliability_dynamics_rows)
    handoff_trials.to_csv(handoff_path, index=False)

handoff_grouped = handoff_trials.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')
handoff_rule_benchmark = aggregate_binary(handoff_grouped, ['distance', 'cohort_group', 'split', 'rule', 'osd_order'])
runtime_summary = handoff_grouped.groupby(['distance', 'cohort_group', 'split', 'rule', 'osd_order'], as_index=False).agg(mean_bp_iterations=('endpoint', 'mean'), mean_bp_runtime_seconds=('bp_runtime_seconds', 'mean'))
handoff_rule_benchmark = handoff_rule_benchmark.merge(runtime_summary, on=['distance', 'cohort_group', 'split', 'rule', 'osd_order'], validate='one_to_one')
handoff_rule_benchmark['LER'] = 1 - handoff_rule_benchmark.logical_correct

handoff_rule_benchmark.to_csv(reliability_dynamics_data_paths.output('handoff_rule_benchmark_csv'), index=False)

def snapshot_endpoints(trace, metrics):
    final = len(trace.llr) - 1
    ranks = metrics['ranks']
    kendall = np.asarray([float(kendalltau(ranks[t - 1], ranks[t]).statistic) for t in range(1, len(ranks))])
    k = min(reliability_dynamics_config['top_k_values'][1], trace.llr.shape[1])
    jaccard = np.asarray([top_jaccard(np.abs(trace.llr[t - 1]), np.abs(trace.llr[t]), k) for t in range(1, len(trace.llr))])
    window = min(reliability_dynamics_config['snapshot_stability_window'], len(jaccard))
    recent_jaccard = np.asarray([jaccard[max(0, index - window + 1):index + 1].mean() for index in range(len(jaccard))])
    eligible_start = window - 1
    top_k_endpoint = int(np.argmax(recent_jaccard[eligible_start:])) + eligible_start + 1

    return {
        'final': final,
        'first_valid': trace.first_valid if trace.first_valid is not None else final,
        'minimum_message_instability': int(np.argmin(metrics['message_residual'])) + 1,
        'maximum_rank_stability': int(np.argmax(kendall)) + 1,
        'maximum_top_k_stability': top_k_endpoint,
    }

snapshot_selection_path = reliability_dynamics_data_paths.output('snapshot_selection_trials_csv')
if stage_cached([snapshot_selection_path]):
    snapshot_selection_trials = reliability_dynamics_read_csv(snapshot_selection_path)
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_case in tqdm(event_cases.itertuples(index=False), total=len(event_cases), desc='snapshot selection', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, reliability_dynamics_trace = case_material(reliability_dynamics_case)
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        for reliability_dynamics_method, reliability_dynamics_endpoint in snapshot_endpoints(reliability_dynamics_trace, metrics).items():
            for reliability_dynamics_order in [0, 1]:
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, np.abs(reliability_dynamics_trace.llr[reliability_dynamics_endpoint]), reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_case.shot_id,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_context.p,
                    'split': reliability_dynamics_case.split,
                    'snapshot_method': reliability_dynamics_method,
                    'endpoint': reliability_dynamics_endpoint,
                    'osd_order': reliability_dynamics_order,
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'logical_error': 1 - reliability_dynamics_correct,
                    'exact_error_match': exact,
                    'bp_runtime_seconds': reliability_dynamics_trace.runtime_seconds[reliability_dynamics_endpoint],
                })

    snapshot_selection_trials = pd.DataFrame(reliability_dynamics_rows)
    snapshot_selection_trials.to_csv(snapshot_selection_path, index=False)

snapshot_grouped = snapshot_selection_trials.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')
snapshot_selection_benchmark = aggregate_binary(snapshot_grouped, ['distance', 'cohort_group', 'split', 'snapshot_method', 'osd_order'])
snapshot_runtime = snapshot_grouped.groupby(['distance', 'cohort_group', 'split', 'snapshot_method', 'osd_order'], as_index=False).agg(mean_bp_iterations=('endpoint', 'mean'), mean_bp_runtime_seconds=('bp_runtime_seconds', 'mean'))
snapshot_selection_benchmark = snapshot_selection_benchmark.merge(snapshot_runtime, on=['distance', 'cohort_group', 'split', 'snapshot_method', 'osd_order'], validate='one_to_one')
snapshot_selection_benchmark['LER'] = 1 - snapshot_selection_benchmark.logical_correct

snapshot_selection_benchmark.to_csv(reliability_dynamics_data_paths.output('snapshot_selection_benchmark_csv'), index=False)


## Experiment 15: optimize BP as a soft-information generator

Scan BP parameters for their ability to generate useful OSD soft inputs. The preferred setting need not be the one that makes stand-alone BP converge fastest.


In [ ]:
parameter_path = reliability_dynamics_data_paths.output('bp_parameter_trials_csv')
if stage_cached([parameter_path]):
    bp_parameter_trials = reliability_dynamics_read_csv(parameter_path)
else:
    reliability_dynamics_rows = []
    selected_cases = pd.concat([frame.head(reliability_dynamics_config['hyperparameter_case_limit']) for _, frame in event_cases.groupby(['distance', 'split'], sort=True)], ignore_index=True)
    reliability_dynamics_grid = list(product(reliability_dynamics_config['scaling_values'], reliability_dynamics_config['damping_values'], reliability_dynamics_config['schedule_values'], reliability_dynamics_config['iteration_values'], [False, True]))

    for reliability_dynamics_case in tqdm(selected_cases.itertuples(index=False), total=len(selected_cases), desc='BP parameter cases', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context = reliability_dynamics_context_for(int(reliability_dynamics_case.level), float(reliability_dynamics_case.p))
        reliability_dynamics_syndrome = reliability_dynamics_bit_array(reliability_dynamics_case.syndrome, reliability_dynamics_context.H.shape[0])
        reliability_dynamics_truth = reliability_dynamics_bit_array(reliability_dynamics_case.true_logical, reliability_dynamics_context.L.shape[0])
        reliability_dynamics_error = reliability_dynamics_bit_array(reliability_dynamics_case.error, reliability_dynamics_context.H.shape[1])

        for reliability_dynamics_scaling, reliability_dynamics_damping, schedule, iterations, early_stop in reliability_dynamics_grid:
            reliability_dynamics_trace = reliability_dynamics_trace_min_sum(
                reliability_dynamics_context,
                reliability_dynamics_syndrome,
                iterations,
                scaling=reliability_dynamics_scaling,
                damping=reliability_dynamics_damping,
                schedule=schedule,
                early_stop=early_stop,
            )
            reliability_dynamics_endpoint = len(reliability_dynamics_trace.llr) - 1
            reliability_dynamics_bp_valid, reliability_dynamics_bp_correct, bp_exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_trace.hard[reliability_dynamics_endpoint], reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
            reliability_dynamics_common = {
                'shot_id': reliability_dynamics_case.shot_id,
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_context.p,
                'split': reliability_dynamics_case.split,
                'scaling': reliability_dynamics_scaling,
                'damping': reliability_dynamics_damping,
                'schedule': schedule,
                'max_iterations': iterations,
                'early_stop': int(early_stop),
                'executed_iterations': reliability_dynamics_endpoint,
                'runtime_seconds': reliability_dynamics_trace.runtime_seconds[-1],
            }

            reliability_dynamics_rows.append(reliability_dynamics_common | {
                'decoder': 'BP',
                'syndrome_consistent': reliability_dynamics_bp_valid,
                'logical_correct': reliability_dynamics_bp_correct,
                'exact_error_match': bp_exact,
            })

            for reliability_dynamics_order in [0, 1]:
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, np.abs(reliability_dynamics_trace.llr[reliability_dynamics_endpoint]), reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append(reliability_dynamics_common | {
                    'decoder': f'OSD{reliability_dynamics_order}',
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'exact_error_match': exact,
                })

    bp_parameter_trials = pd.DataFrame(reliability_dynamics_rows)
    bp_parameter_trials.to_csv(parameter_path, index=False)

bp_parameter_osd_objective = aggregate_binary(bp_parameter_trials, ['distance', 'split', 'scaling', 'damping', 'schedule', 'max_iterations', 'early_stop', 'decoder'])
parameter_runtime = bp_parameter_trials.groupby(['distance', 'split', 'scaling', 'damping', 'schedule', 'max_iterations', 'early_stop', 'decoder'], as_index=False).agg(mean_executed_iterations=('executed_iterations', 'mean'), mean_runtime_seconds=('runtime_seconds', 'mean'))
bp_parameter_osd_objective = bp_parameter_osd_objective.merge(parameter_runtime, on=['distance', 'split', 'scaling', 'damping', 'schedule', 'max_iterations', 'early_stop', 'decoder'], validate='one_to_one')
bp_parameter_osd_objective['LER'] = 1 - bp_parameter_osd_objective.logical_correct

bp_parameter_osd_objective.to_csv(reliability_dynamics_data_paths.output('bp_parameter_osd_objective_csv'), index=False)
bp_parameter_osd_objective.head()


## Experiment 16: unbiased Monte Carlo and BPOSD--Tesseract gap closure

Evaluate the selected methods on an independent unbiased population. Keep the conditional event-cohort findings separate from paired LER and gap closure.


In [ ]:
unbiased_case_path = reliability_dynamics_data_paths.output('unbiased_cases_csv')
if stage_cached([unbiased_case_path]):
    reliability_dynamics_unbiased_cases = reliability_dynamics_read_csv(unbiased_case_path)
    log(2, {
        'stage': 'unbiased cases',
        'source': 'cache',
        'shots': len(reliability_dynamics_unbiased_cases),
    })
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_level in reliability_dynamics_config['levels']:
        for p_index, reliability_dynamics_p in enumerate(reliability_dynamics_config['unbiased_p_values']):
            reliability_dynamics_context = reliability_dynamics_context_for(reliability_dynamics_level, reliability_dynamics_p)
            sampler = reliability_dynamics_context.dem.compile_sampler(seed=reliability_dynamics_config['random_seed'] + 50021 * reliability_dynamics_level + p_index)
            syndromes, truths, errors = sampler.sample(reliability_dynamics_config['unbiased_shots_per_point'], return_errors=True)
            tess_pred, tess_runtime = tesseract_predictions(reliability_dynamics_context, syndromes)
            standard_decoder = BPOSD(max_iter=reliability_dynamics_config['bp_iterations'], osd_order=reliability_dynamics_config['osd_orders_by_level'][str(reliability_dynamics_level)][-1], need_tracking=True)
            forced_decoder = BPOSD(max_iter=reliability_dynamics_config['bp_iterations'], osd_order=reliability_dynamics_config['osd_orders_by_level'][str(reliability_dynamics_level)][-1], need_tracking=True)
            standard_decoder._init_backend(dem=reliability_dynamics_context.dem)
            forced_decoder._init_backend(dem=reliability_dynamics_context.dem)
            iterator = zip(syndromes, truths, errors, tess_pred)

            for index, (reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error, tess) in enumerate(tqdm(iterator, total=len(syndromes), desc=f'unbiased d={reliability_dynamics_context.distance} p={reliability_dynamics_p:g}', disable=reliability_dynamics_config['print_level'] == 1)):
                reliability_dynamics_syndrome = np.asarray(reliability_dynamics_syndrome, dtype=np.uint8)
                reliability_dynamics_truth = np.asarray(reliability_dynamics_truth, dtype=np.uint8)
                reliability_dynamics_error = np.asarray(reliability_dynamics_error, dtype=np.uint8)
                assert np.array_equal(reliability_dynamics_context.H @ reliability_dynamics_error % 2, reliability_dynamics_syndrome)
                assert np.array_equal(reliability_dynamics_logical_label(reliability_dynamics_context, reliability_dynamics_error), reliability_dynamics_truth)
                started = perf_counter()
                standard = np.asarray(standard_decoder.decode_to_correction(reliability_dynamics_syndrome, force_osd=False), dtype=np.uint8)
                standard_runtime = perf_counter() - started
                plain = np.asarray(standard_decoder.bp_correction, dtype=np.uint8)
                started = perf_counter()
                forced = np.asarray(forced_decoder.decode_to_correction(reliability_dynamics_syndrome, force_osd=True), dtype=np.uint8)
                forced_runtime = perf_counter() - started
                reliability_dynamics_shot_id = f'mc-l{reliability_dynamics_level}-p{p_index}-s{index}'
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_shot_id,
                    'level': reliability_dynamics_level,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_p,
                    'split': split_for(reliability_dynamics_shot_id),
                    'syndrome': reliability_dynamics_bits(reliability_dynamics_syndrome),
                    'true_logical': reliability_dynamics_bits(reliability_dynamics_truth),
                    'error': reliability_dynamics_bits(reliability_dynamics_error),
                    'error_weight': int(reliability_dynamics_error.sum()),
                    'syndrome_weight': int(reliability_dynamics_syndrome.sum()),
                    'channel_prior_llr': json_vector(reliability_dynamics_context.prior_llr),
                    'plain_syndrome_consistent': reliability_dynamics_outcome(reliability_dynamics_context, plain, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[0],
                    'plain_logical_correct': reliability_dynamics_outcome(reliability_dynamics_context, plain, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[1],
                    'plain_exact_error_match': reliability_dynamics_outcome(reliability_dynamics_context, plain, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[2],
                    'standard_bposd_correct': reliability_dynamics_outcome(reliability_dynamics_context, standard, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[1],
                    'standard_exact_error_match': reliability_dynamics_outcome(reliability_dynamics_context, standard, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[2],
                    'forced_osd_correct': reliability_dynamics_outcome(reliability_dynamics_context, forced, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[1],
                    'forced_exact_error_match': reliability_dynamics_outcome(reliability_dynamics_context, forced, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)[2],
                    'tesseract_correct': int(np.array_equal(tess, reliability_dynamics_truth)),
                    'standard_runtime_seconds': standard_runtime,
                    'forced_runtime_seconds': forced_runtime,
                    'tesseract_runtime_seconds': tess_runtime,
                })

    reliability_dynamics_unbiased_cases = pd.DataFrame(reliability_dynamics_rows)
    reliability_dynamics_unbiased_cases.to_csv(unbiased_case_path, index=False)
    log(2, {
        'stage': 'unbiased cases',
        'source': 'experiment',
        'shots': len(reliability_dynamics_unbiased_cases),
    })

unbiased_trial_path = reliability_dynamics_data_paths.output('unbiased_reliability_trials_csv')
if stage_cached([unbiased_trial_path]):
    unbiased_trials = reliability_dynamics_read_csv(unbiased_trial_path)
else:
    reliability_dynamics_rows = []
    for reliability_dynamics_case in tqdm(reliability_dynamics_unbiased_cases.itertuples(index=False), total=len(reliability_dynamics_unbiased_cases), desc='unbiased dynamics', disable=reliability_dynamics_config['print_level'] == 1):
        reliability_dynamics_context = reliability_dynamics_context_for(int(reliability_dynamics_case.level), float(reliability_dynamics_case.p))
        reliability_dynamics_syndrome = reliability_dynamics_bit_array(reliability_dynamics_case.syndrome, reliability_dynamics_context.H.shape[0])
        reliability_dynamics_truth = reliability_dynamics_bit_array(reliability_dynamics_case.true_logical, reliability_dynamics_context.L.shape[0])
        reliability_dynamics_error = reliability_dynamics_bit_array(reliability_dynamics_case.error, reliability_dynamics_context.H.shape[1])
        reliability_dynamics_trace = reliability_dynamics_trace_min_sum(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_config['bp_iterations'])
        metrics = trajectory_metrics(reliability_dynamics_context, reliability_dynamics_trace)
        reliability_dynamics_scores = {'final_abs': np.abs(reliability_dynamics_trace.llr[-1])}

        for family in ['temporal_mean', 'temporal_median', 'instability', 'sign_consistency', 'snr', 'conflict', 'combined']:
            reliability_dynamics_scores[family] = selected_score(reliability_dynamics_context.distance, family, reliability_dynamics_trace, metrics)
        for reliability_dynamics_method, reliability_dynamics_score in reliability_dynamics_scores.items():
            for reliability_dynamics_order in [0, 1]:
                started = perf_counter()
                reliability_dynamics_correction = reliability_dynamics_osd_from_reliability(reliability_dynamics_context, reliability_dynamics_syndrome, reliability_dynamics_score, reliability_dynamics_order)
                valid, reliability_dynamics_correct, exact = reliability_dynamics_outcome(reliability_dynamics_context, reliability_dynamics_correction, reliability_dynamics_syndrome, reliability_dynamics_truth, reliability_dynamics_error)
                reliability_dynamics_rows.append({
                    'shot_id': reliability_dynamics_case.shot_id,
                    'distance': reliability_dynamics_context.distance,
                    'p': reliability_dynamics_context.p,
                    'population': 'unbiased',
                    'method': reliability_dynamics_method,
                    'osd_order': reliability_dynamics_order,
                    'syndrome_consistent': valid,
                    'logical_correct': reliability_dynamics_correct,
                    'logical_error': 1 - reliability_dynamics_correct,
                    'exact_error_match': exact,
                    'bp_iterations': reliability_dynamics_config['bp_iterations'],
                    'runtime_seconds': reliability_dynamics_trace.runtime_seconds[-1] + perf_counter() - started,
                })

        for reliability_dynamics_method, reliability_dynamics_correct, exact, runtime in [('plain_BP', reliability_dynamics_case.plain_logical_correct, reliability_dynamics_case.plain_exact_error_match, np.nan), ('standard_BPOSD', reliability_dynamics_case.standard_bposd_correct, reliability_dynamics_case.standard_exact_error_match, reliability_dynamics_case.standard_runtime_seconds), ('forced_OSD', reliability_dynamics_case.forced_osd_correct, reliability_dynamics_case.forced_exact_error_match, reliability_dynamics_case.forced_runtime_seconds), ('Tesseract', reliability_dynamics_case.tesseract_correct, np.nan, reliability_dynamics_case.tesseract_runtime_seconds)]:
            reliability_dynamics_rows.append({
                'shot_id': reliability_dynamics_case.shot_id,
                'distance': reliability_dynamics_context.distance,
                'p': reliability_dynamics_context.p,
                'population': 'unbiased',
                'method': reliability_dynamics_method,
                'osd_order': -1,
                'syndrome_consistent': np.nan,
                'logical_correct': int(reliability_dynamics_correct),
                'logical_error': 1 - int(reliability_dynamics_correct),
                'exact_error_match': exact,
                'bp_iterations': np.nan,
                'runtime_seconds': runtime,
            })

    unbiased_trials = pd.DataFrame(reliability_dynamics_rows)
    unbiased_trials.to_csv(unbiased_trial_path, index=False)

event_benchmark_input = event_method_trials.merge(MEMBERSHIP, on='shot_id', validate='many_to_many')
unbiased_benchmark_input = unbiased_trials.copy()
unbiased_benchmark_input['cohort_group'] = 'all'
all_method_trials = pd.concat([event_benchmark_input, unbiased_benchmark_input], ignore_index=True, sort=False)
benchmark_rows = []

for reliability_dynamics_key, reliability_dynamics_frame in all_method_trials.groupby(['population', 'distance', 'p', 'cohort_group', 'method', 'osd_order'], dropna=False, sort=True):
    reliability_dynamics_ler, ler_low, ler_high = bootstrap_mean(reliability_dynamics_frame.logical_error.to_numpy(), len(benchmark_rows))
    exact, exact_low, exact_high = bootstrap_mean(reliability_dynamics_frame.exact_error_match.to_numpy(), len(benchmark_rows) + 1000)
    paired_delta, paired_low, paired_high = (np.nan, np.nan, np.nan)

    if reliability_dynamics_key[5] in {0, 1}:
        reliability_dynamics_baseline = all_method_trials[(all_method_trials.population == reliability_dynamics_key[0]) & (all_method_trials.distance == reliability_dynamics_key[1]) & (all_method_trials.p == reliability_dynamics_key[2]) & (all_method_trials.cohort_group == reliability_dynamics_key[3]) & (all_method_trials.method == 'final_abs') & (all_method_trials.osd_order == reliability_dynamics_key[5])][['shot_id', 'logical_error']]
        reliability_dynamics_paired = reliability_dynamics_frame[['shot_id', 'logical_error']].merge(reliability_dynamics_baseline, on='shot_id', suffixes=('', '_baseline'), validate='one_to_one')
        if len(reliability_dynamics_paired):
            paired_delta, paired_low, paired_high = bootstrap_mean(reliability_dynamics_paired.logical_error - reliability_dynamics_paired.logical_error_baseline, len(benchmark_rows) + 2000)

    benchmark_rows.append({
        'population': reliability_dynamics_key[0],
        'distance': reliability_dynamics_key[1],
        'p': reliability_dynamics_key[2],
        'cohort_group': reliability_dynamics_key[3],
        'method': reliability_dynamics_key[4],
        'osd_order': reliability_dynamics_key[5],
        'shots': reliability_dynamics_frame.shot_id.nunique(),
        'LER': reliability_dynamics_ler,
        'LER_CI95_low': ler_low,
        'LER_CI95_high': ler_high,
        'repair_rate': reliability_dynamics_frame.logical_correct.mean() if reliability_dynamics_key[3] == 'B' else np.nan,
        'exact_error_match_rate': exact,
        'exact_CI95_low': exact_low,
        'exact_CI95_high': exact_high,
        'paired_LER_delta_vs_final_abs': paired_delta,
        'paired_delta_CI95_low': paired_low,
        'paired_delta_CI95_high': paired_high,
        'mean_runtime_seconds': reliability_dynamics_frame.runtime_seconds.mean(),
    })

reliability_method_osd_benchmark = pd.DataFrame(benchmark_rows)
reliability_method_osd_benchmark.to_csv(reliability_dynamics_data_paths.output('reliability_method_osd_benchmark_csv'), index=False)
reliability_dynamics_gap_rows = []

for (reliability_dynamics_distance, reliability_dynamics_p), cases in reliability_dynamics_unbiased_cases.groupby(['distance', 'p'], sort=True):
    cases = cases.sort_values('shot_id').reset_index(drop=True)
    reference = {
        'standard_BPOSD': 1 - cases.standard_bposd_correct.mean(),
        'forced_OSD': 1 - cases.forced_osd_correct.mean(),
        'Tesseract': 1 - cases.tesseract_correct.mean(),
    }

    reliability_dynamics_denominator = reference['standard_BPOSD'] - reference['Tesseract']
    trial_group = unbiased_trials[(unbiased_trials.distance == reliability_dynamics_distance) & (unbiased_trials.p == reliability_dynamics_p)]
    reliability_dynamics_gap_ids = set(cases[(cases.standard_bposd_correct == 0) & (cases.tesseract_correct == 1)].shot_id)
    reliability_dynamics_rng = np.random.default_rng(reliability_dynamics_config['random_seed'] + 7919 * int(reliability_dynamics_distance) + int(round(float(reliability_dynamics_p) * 100000000.0)))
    bootstrap_indices = reliability_dynamics_rng.integers(0, len(cases), size=(reliability_dynamics_config['bootstrap_repetitions'], len(cases)), dtype=np.int32)
    standard_error = 1 - cases.standard_bposd_correct.to_numpy(dtype=float)
    tesseract_error = 1 - cases.tesseract_correct.to_numpy(dtype=float)
    bootstrap_denominator = standard_error[bootstrap_indices].mean(axis=1) - tesseract_error[bootstrap_indices].mean(axis=1)

    for (reliability_dynamics_method, reliability_dynamics_order), reliability_dynamics_frame in trial_group.groupby(['method', 'osd_order'], sort=True):
        reliability_dynamics_ler = reliability_dynamics_frame.logical_error.mean()
        reliability_dynamics_repair = reliability_dynamics_frame[reliability_dynamics_frame.shot_id.isin(reliability_dynamics_gap_ids)].logical_correct.mean() if reliability_dynamics_gap_ids else np.nan
        reliability_dynamics_paired = cases[['shot_id']].merge(reliability_dynamics_frame[['shot_id', 'logical_error']], on='shot_id', validate='one_to_one')
        assert np.array_equal(reliability_dynamics_paired.shot_id.to_numpy(), cases.shot_id.to_numpy())
        new_error = reliability_dynamics_paired.logical_error.to_numpy(dtype=float)
        reliability_dynamics_eta = np.nan if reliability_dynamics_denominator <= 0 else (reference['standard_BPOSD'] - reliability_dynamics_ler) / reliability_dynamics_denominator
        bootstrap_eta = np.divide(standard_error[bootstrap_indices].mean(axis=1) - new_error[bootstrap_indices].mean(axis=1), bootstrap_denominator, out=np.full(reliability_dynamics_config['bootstrap_repetitions'], np.nan), where=bootstrap_denominator > 0)
        finite_eta = bootstrap_eta[np.isfinite(bootstrap_eta)] if np.isfinite(reliability_dynamics_eta) else np.asarray([])
        reliability_dynamics_gap_rows.append({
            'distance': reliability_dynamics_distance,
            'p': reliability_dynamics_p,
            'method': reliability_dynamics_method,
            'osd_order': reliability_dynamics_order,
            'shots': len(cases),
            'gap_events': len(reliability_dynamics_gap_ids),
            'LER': reliability_dynamics_ler,
            'standard_BPOSD_LER': reference['standard_BPOSD'],
            'Tesseract_LER': reference['Tesseract'],
            'gap_repair_rate': reliability_dynamics_repair,
            'gap_closure_fraction': reliability_dynamics_eta,
            'gap_closure_CI95_low': np.quantile(finite_eta, 0.025) if len(finite_eta) else np.nan,
            'gap_closure_CI95_high': np.quantile(finite_eta, 0.975) if len(finite_eta) else np.nan,
            'gap_closure_bootstrap_samples': len(finite_eta),
        })

bposd_tesseract_gap_closure = pd.DataFrame(reliability_dynamics_gap_rows)
bposd_tesseract_gap_closure.to_csv(reliability_dynamics_data_paths.output('bposd_tesseract_gap_closure_csv'), index=False)
required_stats = ['osd_success_vs_iteration.csv', 'reliability_rank_dynamics.csv', 'instability_rank_correlation.csv', 'relevant_variable_ranks.csv', 'temporal_mean_reliability.csv', 'temporal_median_reliability.csv', 'instability_penalty_ablation.csv', 'sign_consistency_ablation.csv', 'snr_reliability_ablation.csv', 'conflict_reliability_ablation.csv', 'reliability_method_osd_benchmark.csv', 'handoff_rule_benchmark.csv', 'snapshot_selection_benchmark.csv', 'bp_parameter_osd_objective.csv', 'bposd_tesseract_gap_closure.csv']
assert all((reliability_dynamics_data_paths.output('name', name=name).is_file() for name in required_stats))
bposd_tesseract_gap_closure.head()


## Main figures

Figures are displayed inline and are not saved to disk.


In [ ]:
osd_success_vs_iteration


In [ ]:
relative_osd = osd_success_vs_iteration[(osd_success_vs_iteration.distance == 5) & (osd_success_vs_iteration.cohort_group == 'B') & osd_success_vs_iteration.osd_order.isin([0, 1]) & osd_success_vs_iteration.relative_iteration.between(-15, 20)].copy()
if len(relative_osd):
    figure_data = relative_osd.groupby(['osd_order', 'relative_iteration'], as_index=False, sort=True).agg(successes=('logical_correct', 'sum'), evaluations=('logical_correct', 'size'))
    z95 = 1.959963984540054
    reliability_dynamics_probability = figure_data.successes / figure_data.evaluations
    reliability_dynamics_denominator = 1 + z95 ** 2 / figure_data.evaluations
    figure_data['probability'] = reliability_dynamics_probability
    figure_data['CI95_center'] = (reliability_dynamics_probability + z95 ** 2 / (2 * figure_data.evaluations)) / reliability_dynamics_denominator
    figure_data['CI95_half_width'] = z95 * np.sqrt(reliability_dynamics_probability * (1 - reliability_dynamics_probability) / figure_data.evaluations + z95 ** 2 / (4 * figure_data.evaluations ** 2)) / reliability_dynamics_denominator

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    colors = dict(zip([0, 1], sns.color_palette('colorblind', 2)))
    for reliability_dynamics_order in [0, 1]:
        order_data = figure_data[figure_data.osd_order == reliability_dynamics_order].sort_values('relative_iteration')
        if not len(order_data):
            continue
        reliability_dynamics_x = order_data.relative_iteration.to_numpy(dtype=float)
        center = order_data.CI95_center.to_numpy(dtype=float)
        half_width = order_data.CI95_half_width.to_numpy(dtype=float)

        ax.plot(reliability_dynamics_x, order_data.probability, marker='o', color=colors[reliability_dynamics_order], label=f'OSD{reliability_dynamics_order}')
        ax.fill_between(reliability_dynamics_x, center - half_width, center + half_width, color=colors[reliability_dynamics_order], alpha=0.18)

    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set(
        xlim=(-15, 20),
        ylim=(0, 1),
        xlabel='Relative iteration $t-T_{\\mathrm{valid}}$',
        ylabel='$P(\\mathrm{OSD}_q\\ \\mathrm{correct}\\mid t-T_{\\mathrm{valid}})$',
        title='OSD success around first-valid iteration (d=5, Group B)',
    )
    ax.legend(title=None)
    fig.tight_layout()

    plt.show()

kendall_input = rank_grouped[(rank_grouped.distance == 5) & np.isclose(rank_grouped.p, 0.05) & rank_grouped.cohort_group.isin(['A', 'B', 'C', 'D']) & rank_grouped.kendall_previous.notna()]
kendall_data = kendall_input.groupby(['cohort_group', 'iteration'], as_index=False, sort=True).agg(consecutive_kendall=('kendall_previous', 'mean'), samples=('shot_id', 'nunique'))
if len(kendall_data):
    kendall_data = kendall_data.rename(columns={'cohort_group': 'group'})
    reliability_dynamics_grid = sns.relplot(
        data=kendall_data,
        x='iteration',
        y='consecutive_kendall',
        hue='group',
        hue_order=['A', 'B', 'C', 'D'],
        kind='line',
    )
    reliability_dynamics_grid.set_axis_labels('BP iteration', 'Consecutive Kendall correlation')
    reliability_dynamics_grid.figure.suptitle('Reliability ordering stability (d=5, p=0.05)', y=1.03)

    plt.show()

d5_variables = variable_grouped[(variable_grouped.distance == 5) & np.isclose(variable_grouped.p, 0.05) & variable_grouped.cohort_group.isin(['A', 'B', 'C', 'D'])].copy()
if len(d5_variables):
    sampled_variables = d5_variables.sample(min(5000, len(d5_variables)), random_state=reliability_dynamics_config['random_seed'])
    sampled_variables = sampled_variables.rename(columns={'cohort_group': 'group'})
    sampled_variables['instability_bin'] = pd.qcut(sampled_variables.message_instability, q=10, duplicates='drop')
    assert sampled_variables.instability_bin.nunique() >= 2
    binned_trend = sampled_variables.groupby(['group', 'instability_bin'], observed=True, sort=True).agg(
        message_instability=('message_instability', 'median'),
        rank_median=('rank_volatility', 'median'),
        rank_q25=('rank_volatility', lambda values: values.quantile(0.25)),
        rank_q75=('rank_volatility', lambda values: values.quantile(0.75)),
    ).reset_index()
    rho_rows = []

    for (group, reliability_dynamics_shot_id), reliability_dynamics_frame in d5_variables.groupby(['cohort_group', 'shot_id'], sort=True):
        rho_rows.append({
            'group': group,
            'shot_id': reliability_dynamics_shot_id,
            'rho': reliability_dynamics_safe_corr(reliability_dynamics_frame.message_instability, reliability_dynamics_frame.rank_volatility, 'spearman'),
        })

    rho_table = pd.DataFrame(rho_rows).groupby('group', as_index=False).agg(mean_rho=('rho', 'mean'), valid_shots=('rho', 'count'))
    fig, ax = plt.subplots(1, 1, figsize=(9.2, 4.6))
    colors = dict(zip(['A', 'B', 'C'], sns.color_palette('colorblind', 3)))

    for group in ['A', 'B', 'C']:
        group_data = binned_trend[binned_trend.group == group].sort_values('message_instability')
        if not len(group_data):
            continue
        reliability_dynamics_x = group_data.message_instability.to_numpy(dtype=float)

        ax.plot(reliability_dynamics_x, group_data.rank_median, marker='o', color=colors[group], label=f'Group {group}')
        ax.fill_between(reliability_dynamics_x, group_data.rank_q25, group_data.rank_q75, color=colors[group], alpha=0.08)

    ax.set(xlabel='Message instability (bin median)', ylabel='Rank volatility (median, IQR)', title='Instability-to-ranking transmission')
    ax.legend(title=None)
    fig.suptitle('Message instability versus rank volatility (d=5, p=0.05)', y=1.02)
    fig.tight_layout()

    plt.show()

reliability_methods = ['temporal_mean', 'temporal_median', 'instability', 'sign_consistency', 'snr', 'conflict', 'combined']
method_labels = {
    'temporal_mean': 'temporal mean',
    'temporal_median': 'temporal median',
    'instability': 'instability penalty',
    'sign_consistency': 'sign consistency',
    'snr': 'SNR',
    'conflict': 'conflict',
    'combined': 'combined',
}

unbiased_benchmark = reliability_method_osd_benchmark[(reliability_method_osd_benchmark.population == 'unbiased') & reliability_method_osd_benchmark.osd_order.isin([0, 1]) & (reliability_method_osd_benchmark.distance == 5) & reliability_method_osd_benchmark.method.isin(['final_abs', *reliability_methods])].copy()
if len(unbiased_benchmark):
    final_baseline = unbiased_benchmark[unbiased_benchmark.method == 'final_abs'][['p', 'osd_order', 'LER']].rename(columns={'LER': 'final_LER'})
    assert not final_baseline.duplicated(['p', 'osd_order']).any()
    ler_difference = unbiased_benchmark[unbiased_benchmark.method != 'final_abs'].merge(final_baseline, on=['p', 'osd_order'], validate='many_to_one')
    ler_difference['LER_difference'] = ler_difference.LER - ler_difference.final_LER
    ler_difference['method_label'] = ler_difference.method.map(method_labels)
    assert ler_difference.method_label.notna().all()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    colors = dict(zip(method_labels.values(), sns.color_palette('colorblind', len(method_labels))))
    markers = dict(zip(reliability_methods, ['o', 's', '^', 'D', 'P', 'X', 'v']))

    for reliability_dynamics_order, ax in zip([0, 1], axes):
        panel_data = ler_difference[ler_difference.osd_order == reliability_dynamics_order]
        for reliability_dynamics_method in reliability_methods:
            method_data = panel_data[panel_data.method == reliability_dynamics_method].sort_values('p')
            if len(method_data):
                reliability_dynamics_label = method_labels[reliability_dynamics_method]
                linestyle = '--' if reliability_dynamics_method == 'conflict' else '-'
                ax.plot(
                    method_data.p,
                    method_data.LER_difference,
                    marker=markers[reliability_dynamics_method],
                    linestyle=linestyle,
                    color=colors[reliability_dynamics_label],
                    label=reliability_dynamics_label,
                )

        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set(xlabel='Physical error rate p', ylabel='$\\mathrm{LER}_{\\mathrm{modified}}-\\mathrm{LER}_{\\mathrm{original}}$', title=f'OSD{reliability_dynamics_order}')
        ax.grid(alpha=0.25)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc='upper center',
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 0.925),
    )
    fig.suptitle('LER difference between modified and original reliability (d=5)', y=0.985)
    fig.tight_layout(rect=[0, 0, 1, 0.88])

    plt.show()


In [ ]:
kendall_input = rank_grouped[(rank_grouped.distance == 5) & np.isclose(rank_grouped.p, 0.01) & rank_grouped.cohort_group.isin(['A', 'B', 'C', 'D']) & rank_grouped.kendall_previous.notna()]
kendall_data = kendall_input.groupby(['cohort_group', 'iteration'], as_index=False, sort=True).agg(consecutive_kendall=('kendall_previous', 'mean'), samples=('shot_id', 'nunique'))
if len(kendall_data):
    kendall_data = kendall_data.rename(columns={'cohort_group': 'group'})
    reliability_dynamics_grid = sns.relplot(
        data=kendall_data,
        x='iteration',
        y='consecutive_kendall',
        hue='group',
        hue_order=['A', 'B', 'C', 'D'],
        kind='line',
    )
    reliability_dynamics_grid.set_axis_labels('BP iteration', 'Consecutive Kendall correlation')
    reliability_dynamics_grid.figure.suptitle('Reliability ordering stability (d=5, p=0.01)', y=1.03)

    plt.show()

d5_variables = variable_grouped[(variable_grouped.distance == 5) & np.isclose(variable_grouped.p, 0.01) & variable_grouped.cohort_group.isin(['A', 'B', 'C', 'D'])].copy()
if len(d5_variables):
    sampled_variables = d5_variables.sample(min(5000, len(d5_variables)), random_state=reliability_dynamics_config['random_seed'])
    sampled_variables = sampled_variables.rename(columns={'cohort_group': 'group'})
    sampled_variables['instability_bin'] = pd.qcut(sampled_variables.message_instability, q=10, duplicates='drop')
    assert sampled_variables.instability_bin.nunique() >= 2
    binned_trend = sampled_variables.groupby(['group', 'instability_bin'], observed=True, sort=True).agg(
        message_instability=('message_instability', 'median'),
        rank_median=('rank_volatility', 'median'),
        rank_q25=('rank_volatility', lambda values: values.quantile(0.25)),
        rank_q75=('rank_volatility', lambda values: values.quantile(0.75)),
    ).reset_index()
    rho_rows = []

    for (group, reliability_dynamics_shot_id), reliability_dynamics_frame in d5_variables.groupby(['cohort_group', 'shot_id'], sort=True):
        rho_rows.append({
            'group': group,
            'shot_id': reliability_dynamics_shot_id,
            'rho': reliability_dynamics_safe_corr(reliability_dynamics_frame.message_instability, reliability_dynamics_frame.rank_volatility, 'spearman'),
        })

    rho_table = pd.DataFrame(rho_rows).groupby('group', as_index=False).agg(mean_rho=('rho', 'mean'), valid_shots=('rho', 'count'))
    fig, (ax, table_ax) = plt.subplots(1, 2, figsize=(9.2, 4.6), gridspec_kw={'width_ratios': [4, 1.45]})
    colors = dict(zip(['A', 'B', 'C'], sns.color_palette('colorblind', 3)))

    for group in ['A', 'B', 'C']:
        group_data = binned_trend[binned_trend.group == group].sort_values('message_instability')
        if not len(group_data):
            continue
        reliability_dynamics_x = group_data.message_instability.to_numpy(dtype=float)

        ax.plot(reliability_dynamics_x, group_data.rank_median, marker='o', color=colors[group], label=f'Group {group}')
        ax.fill_between(reliability_dynamics_x, group_data.rank_q25, group_data.rank_q75, color=colors[group], alpha=0.18)

    ax.set(xlabel='Message instability (bin median)', ylabel='Rank volatility (median, IQR)', title='Instability-to-ranking transmission')
    ax.legend(title=None)
    table_ax.axis('off')
    table_ax.set_title('Per-shot Spearman $\\rho$', fontsize=10)
    table_ax.table(
        cellText=[[row.group, f'{row.mean_rho:.3f}', int(row.valid_shots)] for row in rho_table.itertuples(index=False)],
        colLabels=['Group', 'Mean $\\rho$', 'Shots'],
        cellLoc='center',
        loc='center',
    )
    fig.suptitle('Message instability versus rank volatility (d=5, p=0.01)', y=1.02)
    fig.tight_layout()

    plt.show()


In [ ]:
check = rank_grouped[(rank_grouped['distance'] == 5) & np.isclose(rank_grouped['p'], 0.01)]
print(check.groupby('cohort_group')['shot_id'].nunique() if 'shot_id' in check.columns else check.groupby('cohort_group').size())


In [ ]:
kendall_input = rank_grouped[(rank_grouped.distance == 5) & np.isclose(rank_grouped.p, 0.03) & rank_grouped.cohort_group.isin(['A', 'B', 'C', 'D']) & rank_grouped.kendall_previous.notna()]
kendall_data = kendall_input.groupby(['cohort_group', 'iteration'], as_index=False, sort=True).agg(consecutive_kendall=('kendall_previous', 'mean'), samples=('shot_id', 'nunique'))
if len(kendall_data):
    kendall_data = kendall_data.rename(columns={'cohort_group': 'group'})
    reliability_dynamics_grid = sns.relplot(
        data=kendall_data,
        x='iteration',
        y='consecutive_kendall',
        hue='group',
        hue_order=['A', 'B', 'C', 'D'],
        kind='line',
    )
    reliability_dynamics_grid.set_axis_labels('BP iteration', 'Consecutive Kendall correlation')
    reliability_dynamics_grid.figure.suptitle('Reliability ordering stability (d=5, p=0.03)', y=1.03)

    plt.show()

d5_variables = variable_grouped[(variable_grouped.distance == 5) & np.isclose(variable_grouped.p, 0.03) & variable_grouped.cohort_group.isin(['A', 'B', 'C', 'D'])].copy()
if len(d5_variables):
    sampled_variables = d5_variables.sample(min(5000, len(d5_variables)), random_state=reliability_dynamics_config['random_seed'])
    sampled_variables = sampled_variables.rename(columns={'cohort_group': 'group'})
    sampled_variables['instability_bin'] = pd.qcut(sampled_variables.message_instability, q=10, duplicates='drop')
    assert sampled_variables.instability_bin.nunique() >= 2
    binned_trend = sampled_variables.groupby(['group', 'instability_bin'], observed=True, sort=True).agg(
        message_instability=('message_instability', 'median'),
        rank_median=('rank_volatility', 'median'),
        rank_q25=('rank_volatility', lambda values: values.quantile(0.25)),
        rank_q75=('rank_volatility', lambda values: values.quantile(0.75)),
    ).reset_index()
    rho_rows = []

    for (group, reliability_dynamics_shot_id), reliability_dynamics_frame in d5_variables.groupby(['cohort_group', 'shot_id'], sort=True):
        rho_rows.append({
            'group': group,
            'shot_id': reliability_dynamics_shot_id,
            'rho': reliability_dynamics_safe_corr(reliability_dynamics_frame.message_instability, reliability_dynamics_frame.rank_volatility, 'spearman'),
        })

    rho_table = pd.DataFrame(rho_rows).groupby('group', as_index=False).agg(mean_rho=('rho', 'mean'), valid_shots=('rho', 'count'))
    fig, (ax, table_ax) = plt.subplots(1, 2, figsize=(9.2, 4.6), gridspec_kw={'width_ratios': [4, 1.45]})
    colors = dict(zip(['A', 'B', 'C'], sns.color_palette('colorblind', 3)))

    for group in ['A', 'B', 'C']:
        group_data = binned_trend[binned_trend.group == group].sort_values('message_instability')
        if not len(group_data):
            continue
        reliability_dynamics_x = group_data.message_instability.to_numpy(dtype=float)

        ax.plot(reliability_dynamics_x, group_data.rank_median, marker='o', color=colors[group], label=f'Group {group}')
        ax.fill_between(reliability_dynamics_x, group_data.rank_q25, group_data.rank_q75, color=colors[group], alpha=0.18)

    ax.set(xlabel='Message instability (bin median)', ylabel='Rank volatility (median, IQR)', title='Instability-to-ranking transmission')
    ax.legend(title=None)
    table_ax.axis('off')
    table_ax.set_title('Per-shot Spearman $\\rho$', fontsize=10)
    table_ax.table(
        cellText=[[row.group, f'{row.mean_rho:.3f}', int(row.valid_shots)] for row in rho_table.itertuples(index=False)],
        colLabels=['Group', 'Mean $\\rho$', 'Shots'],
        cellLoc='center',
        loc='center',
    )
    fig.suptitle('Message instability versus rank volatility (d=5, p=0.03)', y=1.02)
    fig.tight_layout()

    plt.show()


In [ ]:
figure_data = osd_success_vs_iteration[osd_success_vs_iteration.cohort_group == 'B']
if len(figure_data):
    sns.relplot(
        data=figure_data,
        x='relative_iteration',
        y='logical_correct',
        hue='osd_order',
        col='distance',
        kind='line',
        marker='o',
    )
    plt.suptitle('OSD success relative to first syndrome-consistent BP iteration', y=1.03)
    plt.show()
kendall_data = reliability_rank_dynamics[reliability_rank_dynamics.metric == 'kendall_previous']

if len(kendall_data):
    sns.relplot(
        data=kendall_data,
        x='iteration',
        y='mean',
        hue='cohort_group',
        col='distance',
        kind='line',
    )
    plt.suptitle('Consecutive reliability-rank Kendall correlation', y=1.03)
    plt.show()

if len(variable_grouped):
    sampled_variables = variable_grouped.sample(min(5000, len(variable_grouped)), random_state=reliability_dynamics_config['random_seed'])
    sns.relplot(
        data=sampled_variables,
        x='message_instability',
        y='rank_volatility',
        hue='cohort_group',
        col='distance',
        kind='scatter',
        alpha=0.35,
    )
    plt.suptitle('Message instability versus reliability-rank volatility', y=1.03)

    plt.show()

final_relevant = relevant_trials[relevant_trials.iteration == reliability_dynamics_config['snapshot_iterations'][-1]]
if len(final_relevant):
    sns.catplot(
        data=final_relevant,
        x='comparison_group',
        y='mean_rank',
        hue='relevant_set',
        col='distance',
        kind='box',
    )
    plt.suptitle('Rank of error- and repair-relevant variables', y=1.03)
    plt.show()
unbiased_benchmark = reliability_method_osd_benchmark[(reliability_method_osd_benchmark.population == 'unbiased') & reliability_method_osd_benchmark.osd_order.isin([0, 1])]

if len(unbiased_benchmark):
    sns.relplot(
        data=unbiased_benchmark,
        x='p',
        y='LER',
        hue='method',
        style='osd_order',
        col='distance',
        kind='line',
        marker='o',
        height=6,
        aspect=1.5,
    )
    plt.yscale('log')
    plt.suptitle('Dynamics-aware reliability under OSD0/OSD1', y=1.03)

    plt.show()

gap_plot = bposd_tesseract_gap_closure[bposd_tesseract_gap_closure.method.isin(['final_abs', 'combined', 'forced_OSD', 'Tesseract'])]
if len(gap_plot):
    sns.catplot(
        data=gap_plot,
        x='p',
        y='gap_closure_fraction',
        hue='method',
        col='distance',
        kind='bar',
    )
    plt.axhline(0, color='black', linewidth=0.8)
    plt.suptitle('BPOSD--Tesseract gap closure', y=1.03)

    plt.show()


In [ ]:
figure_data = osd_success_vs_iteration[(osd_success_vs_iteration.cohort_group == 'B') & (osd_success_vs_iteration.distance == 5)]
if len(figure_data):
    sns.relplot(
        data=figure_data,
        x='relative_iteration',
        y='logical_correct',
        hue='osd_order',
        kind='line',
        marker='o',
    )
    plt.suptitle('OSD success relative to first syndrome-consistent BP iteration', y=1.03)
    plt.show()
kendall_data = reliability_rank_dynamics[(reliability_rank_dynamics.metric == 'kendall_previous') & (reliability_rank_dynamics.distance == 5)]

if len(kendall_data):
    kendall_data = kendall_data.rename(columns={'cohort_group': 'group'})
    sns.relplot(
        data=kendall_data,
        x='iteration',
        y='mean',
        hue='group',
        kind='line',
    )
    plt.suptitle('Consecutive reliability-rank Kendall correlation', y=1.03)

    plt.show()

d5_variables = variable_grouped[variable_grouped.distance == 5]
if len(d5_variables):
    sampled_variables = d5_variables.sample(min(5000, len(d5_variables)), random_state=reliability_dynamics_config['random_seed'])
    sampled_variables = sampled_variables.rename(columns={'cohort_group': 'group'})
    sns.relplot(
        data=sampled_variables,
        x='message_instability',
        y='rank_volatility',
        hue='group',
        kind='scatter',
        alpha=0.35,
    )
    plt.suptitle('Message instability versus reliability-rank volatility', y=1.03)

    plt.show()

final_relevant = relevant_trials[(relevant_trials.iteration == reliability_dynamics_config['snapshot_iterations'][-1]) & (relevant_trials.distance == 5)]
if len(final_relevant):
    sns.catplot(
        data=final_relevant,
        x='comparison_group',
        y='mean_rank',
        hue='relevant_set',
        kind='box',
    )
    plt.suptitle('Rank of error- and repair-relevant variables', y=1.03)
    plt.show()
unbiased_benchmark = reliability_method_osd_benchmark[(reliability_method_osd_benchmark.population == 'unbiased') & reliability_method_osd_benchmark.osd_order.isin([0, 1]) & (reliability_method_osd_benchmark.distance == 5)]

if len(unbiased_benchmark):
    sns.relplot(
        data=unbiased_benchmark,
        x='p',
        y='LER',
        hue='method',
        style='osd_order',
        kind='line',
        marker='o',
        height=6,
        aspect=1.5,
    )
    plt.yscale('log')
    plt.suptitle('Dynamics-aware reliability under OSD0/OSD1', y=1.03)

    plt.show()

gap_plot = bposd_tesseract_gap_closure[bposd_tesseract_gap_closure.method.isin(['final_abs', 'combined', 'forced_OSD', 'Tesseract']) & (bposd_tesseract_gap_closure.distance == 5)]
if len(gap_plot):
    sns.catplot(
        data=gap_plot,
        x='p',
        y='gap_closure_fraction',
        hue='method',
        kind='bar',
    )
    plt.axhline(0, color='black', linewidth=0.8)
    plt.suptitle('BPOSD--Tesseract gap closure', y=1.03)

    plt.show()


## Interpretation checklist

- H1 is supported only if OSD succeeds at negative relative iteration on group B.
- H2 is supported only if rank stability and message residuals have visibly
  different convergence times.
- H3 is supported only if unstable/sign-inconsistent variables are systematically
  mis-ranked or repair-relevant.
- H4 is supported only if a dynamics-aware method improves held-out OSD0/OSD1
  and the improvement remains in the unbiased Monte Carlo cohort.
- `exact_error_match_rate` and logical-sector correctness are both reported;
  the latter remains the decoder success criterion under degeneracy.

